# Lab 2 - Data Foundations for AI Systems
### Day 1 | AI Solution Architect Programme

> *"Models are rented. Data is owned."* Most AI programmes that fail do not fail because the model was weak -
> they fail because the data underneath was late, wrong, ungoverned, or computed differently in production than in training.

**Scenario for the whole lab: ACME Insurance.** ACME sells motor, home, health and travel cover. Claims arrive from
core systems (tables), mobile apps (JSON), adjusters (free text), photos (pixels) and connected cars (event streams).
Leadership wants a *fraud-triage* AI product. You are the solution architect. Before anyone trains anything, you must
decide **what data lands where, how fresh it is, how good it is, who may see it, and how the model will see the same
data at training time and at serving time.**

| Part | Topic | Architect's question |
|---|---|---|
| 1 | Three kinds of data | What shape is it, and where should it live? |
| 2 | Batch vs real-time pipelines | How fresh must it be, and what does freshness cost? |
| 3 | Feature engineering | How do we turn raw data into signal *without* skew or leakage? |
| 4 | Data quality | How do we know the data is fit for purpose - automatically? |
| 5 | Labeling | Where does ground truth come from, and how noisy is it? |
| 6 | Governance | Who owns it, who can see it, what is personal? |
| 7 | Reference architecture | How do the pieces fit together? (+ a design exercise) |
| 8 | Wrap-up | The 10 questions to ask about any AI data foundation |

### Learning outcomes
By the end you will be able to:
1. Classify data as structured / semi-structured / unstructured / streaming and pick storage and processing for each.
2. Build and compare a batch ETL and a windowed streaming pipeline, and reason about late data, idempotency and delivery guarantees.
3. Engineer features, and *demonstrate* train/serve skew and point-in-time leakage - the two silent killers.
4. Write automated data-quality rules and quantify what bad data costs a model.
5. Measure annotator agreement, aggregate noisy labels, and use active learning to spend labelling budget wisely.
6. Detect and protect PII, check k-anonymity, and express access policy as code.
7. Draw and defend a data architecture for an AI workload with a latency SLA.

**How to use this notebook:** run cells top to bottom. Cells titled *Your turn* are exercises - fill in the stub; the
instructor will reveal the solution. Everything runs offline in a couple of minutes using only numpy, pandas, scikit-learn,
matplotlib and scipy.

## 0. Environment check and shared helpers

Fixed seeds make every run reproducible - a small governance habit in itself (*"can you reproduce last quarter's model?"*
is an audit question). We also set a consistent chart palette so outputs look like one product.

In [ ]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v, "1")        # small data: single-threaded BLAS is faster and avoids thread-oversubscription on shared machines
import sys, platform, json, re, time, math, random, sqlite3, tempfile, hashlib, hmac, itertools, warnings
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn, scipy

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ACME palette: ink, teal, lime, coral, gold
INK, TEAL, LIME, CORAL, GOLD = "#0B1F2A", "#0E7C86", "#9BD13B", "#E4572E", "#F2B134"
plt.rcParams.update({
    "figure.dpi": 100, "axes.edgecolor": INK, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK, "ytick.color": INK, "axes.titleweight": "bold", "axes.titlesize": 12,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
    "axes.prop_cycle": matplotlib.cycler(color=[TEAL, CORAL, GOLD, LIME, INK]),
    "figure.facecolor": "white",
})
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 70)

ART = Path("artifacts/lab2"); ART.mkdir(parents=True, exist_ok=True)   # relative to the labs folder

print(f"python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | "
      f"sklearn {sklearn.__version__} | matplotlib {matplotlib.__version__} | scipy {scipy.__version__}")
print("artifacts will be written to:", ART.resolve().relative_to(Path.cwd().resolve()))

### The ACME data generator

We need realistic, *messy-enough* data and we cannot download any. The function below synthesises ~4,000 claims with
structure that we will deliberately exploit later (e.g. fraud is more likely when the claim is large relative to the
sum insured, filed at night, soon after a previous claim, or for theft). It also writes a free-text adjuster note for each
claim, containing contact details - which will matter in the governance section.

You do not need to read it line by line. The architect's takeaway: **this single table is the "system of record" for the lab**;
everything else (JSON, streams, features, labels) is derived from or sits beside it.

In [ ]:
FIRST = ["Aarav", "Priya", "Rahul", "Sneha", "Vikram", "Ananya", "Karan", "Meera", "Rohan", "Isha", "Arjun", "Divya", "Sanjay", "Neha", "Imran", "Pooja"]
LAST = ["Sharma", "Iyer", "Patel", "Reddy", "Khan", "Nair", "Gupta", "Singh", "Das", "Menon"]
LOSS_TYPES = ["motor_collision", "motor_theft", "home_water", "home_fire", "health_outpatient", "travel_baggage"]
REGIONS = ["North", "South", "East", "West", "Central"]
LOSS_PHRASE = {"motor_collision": "a vehicle collision", "motor_theft": "theft of the insured vehicle", "home_water": "water damage at the insured home",
               "home_fire": "a fire at the insured home", "health_outpatient": "an outpatient treatment claim", "travel_baggage": "lost baggage on a trip"}

def make_acme_claims(n=4000, seed=7):
    """Return (claims, policies). Synthetic, but with realistic structure and a hidden fraud signal."""
    r = np.random.default_rng(seed)
    n_cust = int(n * 0.6)
    cust = pd.DataFrame({
        "customer_id": [f"C{i:05d}" for i in range(n_cust)],
        "customer_name": [f"{r.choice(FIRST)} {r.choice(LAST)}" for _ in range(n_cust)],
        "customer_age": r.integers(19, 80, n_cust),
        "region": r.choice(REGIONS, n_cust, p=[.22, .28, .15, .25, .10]),
        "tenure_years": np.round(np.clip(r.gamma(2.0, 2.0, n_cust), 0.1, 25), 1),
        "sum_insured": np.round(r.lognormal(np.log(600_000), 0.6, n_cust), -3),
    })
    cust["policy_id"] = "P" + cust.customer_id.str[1:]
    cust["premium"] = np.round(cust.sum_insured * r.uniform(0.01, 0.03, n_cust), 0)
    cust["phone"] = [f"+91 {r.integers(70000, 99999)}{r.integers(10000, 99999)}" for _ in range(n_cust)]
    cust["email"] = [f"{nm.lower().replace(' ', '.')}{i % 97}@example.com" for i, nm in enumerate(cust.customer_name)]
    cust["policy_start"] = pd.Timestamp("2024-01-01") - pd.to_timedelta((cust.tenure_years * 365).round(), unit="D")

    # who claims: heavy-tailed, so some customers claim repeatedly
    w = r.gamma(0.8, size=n_cust); w /= w.sum()
    c = cust.iloc[r.choice(n_cust, n, p=w)].reset_index(drop=True)

    hour_w = np.array([.6]*5 + [1, 2, 3, 4] + [5]*8 + [4, 4, 3, 2, 2, 1.5, 1]); assert len(hour_w) == 24
    hour = r.choice(24, n, p=hour_w / hour_w.sum())
    ts = (pd.Timestamp("2024-01-01") + pd.to_timedelta(r.integers(0, 540, n), unit="D")
          + pd.to_timedelta(hour, unit="h") + pd.to_timedelta(r.integers(0, 3600, n), unit="s"))
    c["claim_ts"] = ts                                     # local time (IST)
    c["loss_type"] = r.choice(LOSS_TYPES, n, p=[.35, .08, .15, .07, .25, .10])
    c = c.sort_values("claim_ts").reset_index(drop=True)
    c["claim_id"] = [f"CLM-{i:06d}" for i in range(1, n + 1)]
    c["hour_reported"] = c.claim_ts.dt.hour

    # history (strictly backward-looking => point-in-time correct by construction)
    g = c.groupby("customer_id")
    c["prior_claims"] = g.cumcount()
    c["prev_claim_ts"] = g["claim_ts"].shift(1)
    gap_days = (c.claim_ts - c.prev_claim_ts).dt.total_seconds() / 86400
    t = c.claim_ts.values; c90 = np.zeros(n, int)
    for _, idx in g.indices.items():
        tt = t[idx]; c90[idx] = np.arange(len(idx)) - np.searchsorted(tt, tt - np.timedelta64(90, "D"), side="left")
    c["claims_prior_90d"] = c90

    # amount: fraction of the sum insured (theft/fire are bigger)
    factor = c.loss_type.map({"motor_theft": 2.4, "home_fire": 1.8}).fillna(1.0).values
    ratio = np.clip(r.beta(1.2, 6, n) * factor, 0.002, 0.98)
    c["claim_amount"] = np.maximum(np.round(ratio * c.sum_insured, -2), 2000)
    ratio = c.claim_amount / c.sum_insured

    # hidden fraud process (this is what the AI must discover)
    night = ((c.hour_reported >= 22) | (c.hour_reported <= 4)).values
    quick = (gap_days < 45).fillna(False).values
    logit = (-4.9 + 6.0 * ratio.values + 1.5 * night + 1.2 * quick + 1.0 * (c.loss_type == "motor_theft").values
             + 0.5 * (c.prior_claims >= 2).values + 0.5 * (c.customer_age < 26).values + 0.4 * (c.region == "West").values)
    c["is_fraud"] = (r.random(n) < 1 / (1 + np.exp(-logit))).astype(int)

    # adjuster free-text notes; fraud claims leak a few telltale phrases (noisily)
    fr = c.is_fraud.values
    no_wit = r.random(n) < np.where(fr == 1, .35, .12); cash = r.random(n) < np.where(fr == 1, .20, .06)
    has_fir = r.random(n) < np.where(fr == 1, .45, .60)
    extra_fraud = ["Vehicle was repaired before inspection could be arranged.", "Claimant was reluctant to share original invoices.", "Policy cover was increased shortly before the loss."]
    extra_ok = ["Claimant cooperated fully and shared documents promptly.", "Inspection completed on the same day.", "Photos and invoices were uploaded via the mobile app."]
    notes = []
    for i, row in enumerate(c.itertuples()):
        s = [f"FNOL {row.claim_ts:%Y-%m-%d}: insured {row.customer_name} reports {LOSS_PHRASE[row.loss_type]}.",
             f"Estimated damage Rs {row.claim_amount:,.0f}."]
        s.append(f"FIR-{row.claim_ts:%Y}-{r.integers(100, 9999):04d} filed with local police." if has_fir[i] else "FIR not filed yet.")
        if no_wit[i]: s.append("No witnesses available.")
        if cash[i]: s.append("Claimant asked for settlement in cash.")
        if r.random() < (.40 if fr[i] else .25): s.append(str(r.choice(extra_fraud if fr[i] else extra_ok)))
        s.append(f"Contact {row.phone} / {row.email}.")
        notes.append(" ".join(s))
    c["notes"] = notes
    c["claim_ts_utc"] = c.claim_ts - pd.Timedelta(hours=5, minutes=30)     # what the platform stores
    c["claim_date"] = c.claim_ts.dt.normalize()
    cols = ["claim_id", "policy_id", "customer_id", "customer_name", "phone", "email", "claim_ts", "claim_ts_utc", "claim_date", "hour_reported",
            "loss_type", "claim_amount", "sum_insured", "premium", "customer_age", "region", "tenure_years", "policy_start",
            "prev_claim_ts", "prior_claims", "claims_prior_90d", "notes", "is_fraud"]
    policies = cust[["policy_id", "customer_id", "sum_insured", "premium", "policy_start"]].copy()
    return c[cols], policies

claims, policies = make_acme_claims()
print(f"claims: {claims.shape} | policies: {policies.shape} | fraud rate: {claims.is_fraud.mean():.1%} | "
      f"{claims.claim_date.min():%Y-%m-%d} -> {claims.claim_date.max():%Y-%m-%d}")
claims[["claim_id", "policy_id", "claim_date", "loss_type", "claim_amount", "region", "is_fraud"]].head()

---
# Part 1 - Three (really four) kinds of data

Before choosing a database, ask: **what is the *shape* of the data and how will it be *asked questions*?**
Analogy: a library holds a card catalogue (structured), books (unstructured), a river of newspapers arriving every minute
(streaming), and forms whose fields vary by publisher (semi-structured). You would not shelve them the same way.

We will touch each kind in turn, using ACME's own data, and notice that each one is *queried differently*.

## 1.1 Structured: rows, columns, a schema

Structured data has a schema known **before** you read it. Types are enforced, joins are cheap, and SQL/pandas answer
aggregate questions instantly. This is the bedrock for classical ML (tabular models) and for BI.

In [ ]:
# The schema is the contract: names + types. Everything downstream depends on it.
print(claims[["claim_id", "claim_date", "loss_type", "claim_amount", "customer_age", "region", "is_fraud"]].dtypes.to_string())

# A structured question is a declarative one: "what is the fraud rate and exposure by loss type?"
summary = (claims.groupby("loss_type")
           .agg(claims=("claim_id", "count"), total_inr_m=("claim_amount", lambda s: s.sum() / 1e6),
                avg_amount=("claim_amount", "mean"), fraud_rate=("is_fraud", "mean"))
           .round(3).sort_values("fraud_rate", ascending=False))
summary

## 1.2 Semi-structured: JSON from the mobile app

ACME's app submits a *First Notice of Loss* (FNOL) as JSON. It has structure (keys, nesting) but the **shape varies per record**:
one claimant has no phone, another attaches three photos, a third adds a `flags` list. A rigid table would either explode in
sparse columns or lose data. Document stores / data-lake JSON handle this - but *you* pay the cost at read time ("schema-on-read").

`pandas.json_normalize` flattens nested JSON into columns; `record_path` explodes repeating groups (the attachments) into rows.

In [ ]:
fnol_docs = [
    {"claim_id": "CLM-900001", "channel": "mobile_app",
     "reporter": {"name": "Aarav Sharma", "contact": {"phone": "+91 9876501234"}},
     "loss": {"type": "motor_collision", "estimated_amount": 84000, "location": {"city": "Pune", "geo": {"lat": 18.52, "lon": 73.86}}},
     "attachments": [{"kind": "photo", "uri": "s3://acme-raw/photos/900001_1.jpg", "kb": 812}, {"kind": "photo", "uri": "s3://acme-raw/photos/900001_2.jpg", "kb": 640}]},
    {"claim_id": "CLM-900002", "channel": "call_centre",
     "reporter": {"name": "Priya Iyer"},                                     # <- no contact block at all
     "loss": {"type": "home_water", "estimated_amount": 152000, "location": {"city": "Chennai"}},
     "attachments": []},
    {"claim_id": "CLM-900003", "channel": "mobile_app", "flags": ["third_party_involved", "injury_reported"],
     "reporter": {"name": "Rahul Patel", "contact": {"phone": "+91 9123456780"}},
     "loss": {"type": "motor_collision", "estimated_amount": 410000, "location": {"city": "Mumbai", "geo": {"lat": 19.07, "lon": 72.87}}},
     "attachments": [{"kind": "video", "uri": "s3://acme-raw/video/900003_1.mp4", "kb": 20480}]},
]
flat = pd.json_normalize(fnol_docs)                      # one row per claim; nested keys become dotted column names
print("Flattened columns:", list(flat.columns), "\n")
print(flat[["claim_id", "reporter.name", "reporter.contact.phone", "loss.type", "loss.estimated_amount", "loss.location.city"]].to_string(index=False))

# repeating group -> its own table (1 claim : many attachments). This is the 'normalisation' step lakehouses do for you.
att = pd.json_normalize(fnol_docs, record_path="attachments", meta=["claim_id", "channel"])
print("\nAttachments table:\n", att.to_string(index=False))
print("\nNote the NaN phone for CLM-900002 and the missing 'flags' column for two docs: schema-on-read means YOU handle missingness.")

## 1.3 Unstructured: free text and pixels

Roughly 80% of enterprise data is unstructured: emails, PDFs, call transcripts, photos. It has *no schema you can query* - but it is
often where the most valuable signal lives. Two examples from ACME:

1. **Adjuster notes** (text). We can extract fields with regular expressions - brittle, but it shows the idea of *structure-from-unstructured*.
2. **A damage photo** (pixels). To a machine an image is just a numeric array. Here we fabricate a tiny 28x28 greyscale "car panel" with a dent.

**Why this matters for later today:** regex works on *templated* text. The moment an adjuster writes "bumper got hit pretty badly, ~85k I reckon",
regex breaks - and that gap is exactly where LLMs and embeddings (Lab 3) earn their keep.

In [ ]:
note = claims.loc[claims.is_fraud == 1, "notes"].iloc[3]
print("RAW NOTE:\n", re.sub(r"(.{85}\s)", r"\1\n   ", note), "\n")

# 'Structure from unstructured' with regexes: pull out amount, FIR number, phone, boolean flags.
PATTERNS = {
    "amount_inr": r"Rs\s*([\d,]+)",
    "fir_no": r"(FIR-\d{4}-\d{4})",
    "phone": r"(\+91\s?\d{10})",
    "no_witnesses": r"[Nn]o witnesses",
    "cash_request": r"in cash",
}
def extract_fields(text):
    out = {}
    for k, pat in PATTERNS.items():
        m = re.search(pat, text)
        out[k] = (m.group(1) if m and m.groups() else bool(m))
    out["amount_inr"] = int(out["amount_inr"].replace(",", "")) if out["amount_inr"] else None
    return out

extracted = pd.DataFrame([extract_fields(t) for t in claims.notes.head(500)])
print(pd.Series(extract_fields(note)).to_string())
# validation: does the text-derived amount reconcile with the structured table? (cross-source consistency = a data quality idea)
match = (extracted.amount_inr.values == claims.claim_amount.head(500).astype(int).values).mean()
print(f"\nText-derived amount matches structured claim_amount for {match:.1%} of 500 notes (templated text => regex works).")

In [ ]:
# A toy 'image' - just numbers. Shape (height, width), values 0..255.
yy, xx = np.mgrid[0:28, 0:28]
panel = 150 + 40 * (xx / 27)                                                 # brushed-metal gradient
dent = 90 * np.exp(-(((xx - 18) ** 2 + (yy - 10) ** 2) / (2 * 3.0 ** 2)))   # gaussian dent
noise = np.random.default_rng(1).normal(0, 4, (28, 28))
photo = np.clip(panel - dent + noise, 0, 255).astype(np.uint8)
reference = np.clip(panel + noise * 0, 0, 255)

# The simplest possible 'feature extractor': deviation from an undamaged reference panel.
damage_map = np.abs(photo.astype(float) - reference)
damage_score = (damage_map > 25).mean()

fig, ax = plt.subplots(1, 2, figsize=(7.5, 3.3))
ax[0].imshow(photo, cmap="gray", vmin=0, vmax=255); ax[0].set_title("Photo: 28x28 uint8 array"); ax[0].grid(False)
im = ax[1].imshow(damage_map, cmap="inferno"); ax[1].set_title(f"Pixel deviation (damage score={damage_score:.2f})"); ax[1].grid(False)
plt.tight_layout(); plt.show()
print(f"dtype={photo.dtype}, shape={photo.shape}, raw size={photo.nbytes} bytes -> a real 12MP phone photo is ~{12e6*3/1e6:.0f} MB uncompressed.")
print("Unstructured -> numbers is the whole game. Deep nets (and embeddings) learn this conversion instead of hand-writing it.")

## 1.4 Streaming: data that never "finishes"

Telematics boxes in insured cars emit events continuously: speed, harsh braking, GPS. There is no "table" - there is an **unbounded sequence**
with **event time** (when it happened) and a different **arrival time** (when we received it). In Python the natural abstraction is a *generator*:
you can't `len()` it, you can only consume it as it flows.

We will reuse this generator in Part 2 to build a streaming pipeline.

In [ ]:
def telematics_events(n_events=4800, seed=11, rate=4.0, n_devices=8):
    """Yield telematics events in event-time order. `rate` = average events/second across the fleet."""
    r = random.Random(seed)
    t0 = pd.Timestamp("2025-03-01 09:00:00")
    t = 0.0
    for i in range(n_events):
        t += r.expovariate(rate)                                   # Poisson arrivals
        speed = max(0.0, r.gauss(55, 18))
        yield {"event_id": f"E{i:06d}", "device_id": f"D{r.randrange(n_devices)}", "event_time": t,
               "ts": (t0 + pd.Timedelta(seconds=t)).isoformat(timespec="milliseconds"),
               "speed_kmh": round(speed, 1), "harsh_brake": speed > 85 and r.random() < .3}

stream = telematics_events()
print("type:", type(stream).__name__, "-> lazy; nothing computed yet. Taking the first 5 events:\n")
for ev in itertools.islice(stream, 5):
    print(ev)

## 1.5 The cheat-sheet: type -> storage -> processing -> AI use

| Data type | ACME example | Typical storage | Processing pattern | How AI uses it |
|---|---|---|---|---|
| **Structured** | Claims, policies, payments | Relational DB / warehouse / lakehouse tables (Parquet, Delta, Iceberg) | SQL, joins, batch ETL | Tabular ML (fraud, pricing, churn); features; BI; ground-truth labels |
| **Semi-structured** | FNOL JSON, API payloads, logs | Object store + document DB; lakehouse JSON/variant columns | Schema-on-read, flattening, schema-evolution handling | Parsed into features; LLM tool outputs; event enrichment |
| **Unstructured** | Adjuster notes, PDFs, photos, call audio | Object store (S3/ADLS/GCS) + metadata catalogue; later a vector index | OCR/ASR, NLP, computer vision, embeddings | LLM/RAG, document AI, vision models, semantic search (Lab 3) |
| **Streaming** | Telematics, card swipes, clickstream | Log/bus (Kafka, Kinesis, Event Hubs) -> hot store + lake sink | Windowed aggregation, CEP, stateful processing | Real-time scoring, anomaly detection, online features |

**Architect's rule of thumb:** *store raw things raw (cheap object storage), model curated things as tables, and keep streams on a durable log
so you can replay.* Raw is your insurance policy: when the business asks a new question in 2 years, you still have the original evidence.

### Your turn 1 - extract one more field
Adjuster notes sometimes mention a third kind of evidence. Write a regex that extracts **the claim year from the FIR number** (the `2024` in `FIR-2024-0123`)
and apply it to the first 200 notes. Report how many notes have *no* FIR (those will be `None`).

In [ ]:
# TODO: write a regex with ONE capture group for the 4-digit year inside the FIR number.
# HINT: re.search(r"FIR-(\d{4})-", text) returns a match object or None - check before calling .group(1).
fir_year_pattern = None   # <- replace with your pattern

def fir_year(text):
    pass   # <- return int year or None

# years = pd.Series([fir_year(t) for t in claims.notes.head(200)])
# print(years.value_counts(dropna=False))

---
# Part 2 - Data pipelines: batch vs real-time

A pipeline is a repeatable, observable way to move data from where it is *created* to where it is *useful*. Two questions define the design:

* **Freshness:** how old may the data be when someone (or a model) uses it? *Nightly? Minutes? Milliseconds?*
* **Correctness:** what happens when data is late, duplicated, malformed, or replayed?

Analogy: **batch is the monthly bank statement; streaming is the SMS alert.** The statement is complete, reconciled and cheap. The alert is instant but
must be sent before you know everything. Mature banks run both - and so will you.

## 2.1 Batch ETL: extract -> validate -> transform -> load

We simulate a nightly file from the core claims system landing as CSV. Real life is never clean, so we sneak a few bad rows in:
negative amounts, a missing policy id, and an accidental duplicate. The pipeline has four small, *testable* steps - the shape of nearly every
batch job, whether it runs in Airflow, Databricks, Glue or a cron script.

Two architect-grade details to notice:
1. **Quarantine, don't drop.** Rejected rows go to a side table with a reason, so data owners can fix the source.
2. **Idempotent load.** The job can be re-run safely (after a crash, a backfill, a retry) without double-counting - we use a primary key + upsert.

In [ ]:
WORK = Path(tempfile.mkdtemp(prefix="lab2_"))       # temp dir for the landing zone and the SQLite 'warehouse'
RAW_COLS = ["claim_id", "policy_id", "claim_date", "loss_type", "claim_amount", "sum_insured", "region", "customer_age", "phone", "is_fraud"]

# --- simulate the nightly extract file from the source system, with realistic defects -------------------
landing = claims[RAW_COLS].head(1500).copy()
landing.loc[landing.index[[10, 55, 300]], "claim_amount"] = [-5000.0, -120.0, 0.0]      # impossible amounts
landing.loc[landing.index[[20, 21]], "policy_id"] = None                                  # missing key
landing = pd.concat([landing, landing.iloc[[100, 101]]], ignore_index=True)               # accidental duplicates
raw_path = WORK / "claims_2025-06-23.csv"
landing.to_csv(raw_path, index=False)
print(f"landing file: {raw_path.name}, {len(landing)} rows ({raw_path.stat().st_size/1e3:.0f} KB)")

# --- the four steps ---------------------------------------------------------------------------------------
def extract(path):
    return pd.read_csv(path, parse_dates=["claim_date"])

def validate(df):
    """Split into (good, rejected). Every reject carries a human-readable reason."""
    reasons = pd.Series("", index=df.index)
    reasons[df.policy_id.isna()] += "missing_policy_id;"
    reasons[df.claim_amount <= 0] += "non_positive_amount;"
    reasons[~df.loss_type.isin(LOSS_TYPES)] += "unknown_loss_type;"
    reasons[df.duplicated("claim_id", keep="first")] += "duplicate_claim_id;"
    bad = reasons != ""
    return df[~bad].copy(), df[bad].assign(reject_reason=reasons[bad])

def transform(df):
    out = df.copy()
    out["claim_month"] = out.claim_date.dt.strftime("%Y-%m")
    out["line_of_business"] = out.loss_type.str.split("_").str[0]
    out["amount_band"] = pd.cut(out.claim_amount, [0, 50_000, 200_000, 1_000_000, np.inf], labels=["<50k", "50-200k", "200k-1M", ">1M"]).astype(str)
    out["claim_date"] = out.claim_date.dt.strftime("%Y-%m-%d")                 # SQLite has no native date type
    out["ingested_at"] = pd.Timestamp.now().isoformat(timespec="seconds")      # lineage: when did we load this?
    return out

DDL = """CREATE TABLE IF NOT EXISTS claims_curated (
    claim_id TEXT PRIMARY KEY, policy_id TEXT NOT NULL, claim_date TEXT, loss_type TEXT, claim_amount REAL, sum_insured REAL,
    region TEXT, customer_age INTEGER, phone TEXT, is_fraud INTEGER, claim_month TEXT, line_of_business TEXT, amount_band TEXT, ingested_at TEXT)"""

def load(df, conn, rejects):
    conn.execute(DDL)
    cols = ",".join(df.columns)
    conn.executemany(f"INSERT OR REPLACE INTO claims_curated ({cols}) VALUES ({','.join('?' * len(df.columns))})", df.itertuples(index=False, name=None))
    rejects.to_sql("claims_quarantine", conn, if_exists="replace", index=False)
    conn.commit()

def run_batch(path, conn):
    t0 = time.perf_counter()
    raw = extract(path); good, rejected = validate(raw); curated = transform(good); load(curated, conn, rejected)
    return {"read": len(raw), "loaded": len(curated), "quarantined": len(rejected), "seconds": round(time.perf_counter() - t0, 3)}

conn = sqlite3.connect(WORK / "acme_warehouse.db")
print("run 1:", run_batch(raw_path, conn))
print("run 2 (re-run same file):", run_batch(raw_path, conn))
print("rows in table after TWO runs:", conn.execute("SELECT COUNT(*) FROM claims_curated").fetchone()[0], " <- idempotent: no double counting")
pd.read_sql("SELECT claim_id, policy_id, claim_amount, reject_reason FROM claims_quarantine", conn)

### Why SQL still matters
The curated table is now queryable by anyone with SQL - analysts, BI tools, and (increasingly) LLM text-to-SQL agents. That is the real payoff of a
*curated* layer: one trusted place instead of a hundred spreadsheets. Let's ask three business questions. Note the third uses a **window function**,
something we will mimic by hand in the streaming section.

In [ ]:
q1 = """SELECT line_of_business, COUNT(*) AS claims, ROUND(SUM(claim_amount)/1e6,1) AS inr_millions,
               ROUND(100.0*AVG(is_fraud),1) AS fraud_pct
        FROM claims_curated GROUP BY line_of_business ORDER BY inr_millions DESC"""
q2 = """SELECT region, amount_band, COUNT(*) AS n FROM claims_curated GROUP BY region, amount_band ORDER BY region, n DESC LIMIT 6"""
q3 = """SELECT claim_month, claims, SUM(claims) OVER (ORDER BY claim_month) AS cumulative_claims FROM
        (SELECT claim_month, COUNT(*) AS claims FROM claims_curated GROUP BY claim_month)"""
for q in (q1, q2, q3):
    print(pd.read_sql(q, conn).to_string(index=False), "\n")

### Naive loading is a classic production incident
What if the load step had simply *appended*? Retries (very normal in distributed systems) silently double your revenue and your fraud counts.
Dashboards look plausible, nobody notices, and a model trains on duplicated rows. **Idempotency** - "running it twice has the same effect as once" - is the cheapest
reliability feature you will ever buy.

In [ ]:
naive = sqlite3.connect(":memory:")
good, _ = validate(extract(raw_path))
for attempt in range(1, 3):                                                       # job "retried" after a timeout
    good.assign(claim_date=good.claim_date.dt.strftime("%Y-%m-%d")).to_sql("claims_naive", naive, if_exists="append", index=False)
    print(f"naive append, after attempt {attempt}: {naive.execute('SELECT COUNT(*), ROUND(SUM(claim_amount)/1e6,1) FROM claims_naive').fetchone()} (rows, INR m)")
print("idempotent upsert, after any number of attempts:", conn.execute("SELECT COUNT(*), ROUND(SUM(claim_amount)/1e6,1) FROM claims_curated").fetchone())

## 2.2 Streaming: windows, watermarks and late data

A stream has no end, so "total claims" has no meaning - we ask **"how many events *per minute*"**. That requires *windows*:

* **Tumbling window** - fixed, non-overlapping buckets (every 60 s). Each event belongs to exactly one.
* **Sliding (hopping) window** - overlapping (last 5 minutes, refreshed every minute). Each event belongs to several. Great for "rolling" features like *harsh brakes in the last 5 minutes*.

And the hard part: **events arrive late and out of order** (a phone in a tunnel uploads its data 4 minutes later). The system must decide *when to stop waiting* for a window.
That decision is the **watermark**: "I believe I have now seen everything with event time earlier than *W*." A common rule is
`watermark = max_event_time_seen - allowed_lateness`. Once the watermark passes a window's end, we emit the result and close it; anything arriving afterward is *late*
and goes to a side output (to be corrected later by batch).

**The trade-off in one sentence: a bigger lateness allowance gives more complete answers but slower ones.**

First, let us make events arrive out of order: 5% of events are delayed by 30-300 s on top of a small network jitter.

In [ ]:
def with_arrival(events, seed=5, late_frac=0.05):
    """Attach an arrival_time (processing clock) to each event and return them in ARRIVAL order."""
    r = np.random.default_rng(seed); out = []
    for e in events:
        delay = r.exponential(2.0)                                   # normal jitter, ~2 s
        if r.random() < late_frac: delay += r.uniform(30, 300)       # the 'phone in a tunnel' events
        out.append({**e, "arrival_time": e["event_time"] + delay})
    return sorted(out, key=lambda e: e["arrival_time"])

events = with_arrival(telematics_events())
ooo = sum(1 for a, b in zip(events, events[1:]) if b["event_time"] < a["event_time"])
print(f"{len(events)} events spanning {events[-1]['event_time']/60:.1f} min of event time; {ooo} arrive out of order.")

In [ ]:
def run_stream(events, window=60, lateness=30):
    """Tumbling-window aggregation with a watermark. Pure Python; processes events one at a time in arrival order.
    Returns (emitted_windows_df, late_events_list)."""
    state, emitted, late = {}, [], []
    max_et = -math.inf
    def emit(w0, st, clock):
        emitted.append({"window_start": w0, "count": st["n"], "avg_speed": st["spd"] / st["n"], "harsh_brakes": st["hb"],
                        "emit_latency_s": clock - (w0 + window)})           # how long after the window ended did we answer?
    for e in events:
        w0 = int(e["event_time"] // window) * window
        if w0 + window <= max_et - lateness:                                # window already closed by the watermark
            late.append(e)
        else:
            st = state.setdefault(w0, {"n": 0, "spd": 0.0, "hb": 0})
            st["n"] += 1; st["spd"] += e["speed_kmh"]; st["hb"] += int(e["harsh_brake"])
        max_et = max(max_et, e["event_time"])
        watermark = max_et - lateness
        for ws in sorted(state):                                            # close every window the watermark has passed
            if ws + window <= watermark: emit(ws, state.pop(ws), e["arrival_time"])
            else: break
    for ws in sorted(state): emit(ws, state.pop(ws), events[-1]["arrival_time"])       # end-of-stream flush
    return pd.DataFrame(emitted), late

stream_df, late_events = run_stream(events, window=60, lateness=30)
print(f"emitted {len(stream_df)} one-minute windows; {len(late_events)} events arrived after their window closed (side output)")
stream_df.head(6).round(2)

### Batch is the ground truth - how far off is the stream?
Because batch sees *all* data after the fact, it can compute the exact answer by **event time**. Compare per-minute counts from the stream (lateness = 30 s) with the batch truth.
The gap is precisely the late events. This is the heart of the **Lambda architecture**: a fast "speed layer" gives an approximate answer now; a "batch layer" corrects it later.

In [ ]:
ev = pd.DataFrame(events)
truth = ev.assign(window_start=(ev.event_time // 60 * 60).astype(int)).groupby("window_start").agg(true_count=("event_id", "count")).reset_index()
cmp_ = truth.merge(stream_df[["window_start", "count"]].rename(columns={"count": "stream_count"}), on="window_start", how="left")
cmp_["missed"] = cmp_.true_count - cmp_.stream_count
print(f"Total events: truth={cmp_.true_count.sum()}  stream={cmp_.stream_count.sum()}  missed={cmp_.missed.sum()} ({cmp_.missed.sum()/cmp_.true_count.sum():.1%})")
print(f"Windows where stream != truth: {(cmp_.missed != 0).sum()} of {len(cmp_)}")
print(f"Median stream latency: {stream_df.emit_latency_s.median():.0f}s after window close  vs  batch (nightly): ~{12*3600:,}s")

# batch 'correction': reprocess the late side-output and merge -> reconciled counts equal the truth
fix = pd.DataFrame(late_events); fix["window_start"] = (fix.event_time // 60 * 60).astype(int)
recon = cmp_.merge(fix.groupby("window_start").size().rename("late_fix").reset_index(), on="window_start", how="left").fillna({"late_fix": 0})
recon["reconciled"] = recon.stream_count + recon.late_fix
print("After applying the late-event correction, reconciled == truth for every window:", bool((recon.reconciled == recon.true_count).all()))

### Sliding windows from panes
A 5-minute window that advances every minute does not need to re-scan 5 minutes of events: it is just the **sum of the last five 1-minute tumbling "panes"**.
This is how real engines make rolling aggregates cheap. Below, a rolling "harsh-brake count over the last 5 minutes" - the kind of feature an
usage-based-insurance risk model consumes online.

In [ ]:
panes = stream_df.set_index("window_start").sort_index()
slide = pd.DataFrame({"events_5min": panes["count"].rolling(5).sum(), "harsh_brakes_5min": panes["harsh_brakes"].rolling(5).sum()}).dropna()
print("Sliding 5-min window, advancing every 60s (first rows have <5 panes so are skipped):")
print(slide.head(5).astype(int).to_string())

fig, ax = plt.subplots(figsize=(8, 3))
ax.step(panes.index / 60, panes["count"], where="post", color=TEAL, label="tumbling 1-min count")
ax.plot(slide.index / 60, slide.events_5min / 5, color=CORAL, lw=2, label="sliding 5-min (per-minute avg)")
ax.set_xlabel("event time (minutes)"); ax.set_ylabel("events / min"); ax.set_title("Tumbling vs sliding window: sliding smooths the signal"); ax.legend()
plt.tight_layout(); plt.show()

## 2.3 The lateness dial: completeness vs latency
Let us measure the trade-off rather than assert it. We re-run the same stream with different `allowed_lateness` values and record
(a) what share of events made it into a window on time and (b) how long we waited before emitting.

In [ ]:
rows = []
for lat in [0, 10, 30, 60, 120, 300, 600]:
    s, l = run_stream(events, window=60, lateness=lat)
    rows.append({"allowed_lateness_s": lat, "completeness_pct": 100 * (1 - len(l) / len(events)), "median_latency_s": s.emit_latency_s.median()})
tradeoff = pd.DataFrame(rows).round(1); print(tradeoff.to_string(index=False))

fig, ax1 = plt.subplots(figsize=(7.5, 3.4))
ax1.plot(tradeoff.allowed_lateness_s, tradeoff.completeness_pct, "o-", color=TEAL, lw=2); ax1.set_ylabel("% events counted on time", color=TEAL)
ax1.set_xlabel("allowed lateness (seconds)")
ax2 = ax1.twinx(); ax2.plot(tradeoff.allowed_lateness_s, tradeoff.median_latency_s, "s--", color=CORAL, lw=2); ax2.set_ylabel("median emit latency (s)", color=CORAL); ax2.grid(False)
ax2.spines["right"].set_visible(True)
ax1.set_title("The watermark dial: wait longer -> more complete, less fresh"); plt.tight_layout(); plt.show()

## 2.4 Delivery guarantees: at-most-once, at-least-once, exactly-once

Networks fail, so a producer that does not get an acknowledgement *retries*. That yields one of three contracts:

| Guarantee | Meaning | Failure mode | Typical use |
|---|---|---|---|
| **At-most-once** | Fire and forget | Data loss | Metrics, non-critical logs |
| **At-least-once** | Retry until acked | **Duplicates** | Default for most brokers; the safe choice *if consumers are idempotent* |
| **Exactly-once** (effects) | Atomic commit of state + offsets, or dedupe by key | Complexity, throughput cost | Payments, claims ledgers |

Pragmatic truth: "exactly-once" is almost always **at-least-once delivery + idempotent processing** (dedupe on a unique event id). Let's prove it:
redeliver 3% of events and compare a naive counter to a deduplicating one.

In [ ]:
r = random.Random(3)
redelivered = events + [dict(e) for e in events if r.random() < 0.03]               # at-least-once => some duplicates
r.shuffle(redelivered); redelivered.sort(key=lambda e: e["arrival_time"])
naive_count = len(redelivered)
seen, deduped = set(), []
for e in redelivered:                                                                # idempotent consumer: remember event_ids
    if e["event_id"] not in seen: seen.add(e["event_id"]); deduped.append(e)
print(f"true events={len(events)} | naive consumer counts {naive_count} (+{naive_count-len(events)} phantom events) | idempotent consumer counts {len(deduped)}")
print("Dedup state costs memory: in production use a TTL (e.g. 24h) or a transactional sink so 'seen' does not grow forever.")

## 2.5 Choosing: freshness vs cost vs complexity

Fresher data is **not free**: always-on clusters, state management, on-call, harder testing, harder backfills. The numbers below are *illustrative* (simulated, not benchmarks),
but the shape is real: cost and complexity climb steeply as the latency requirement drops below a few minutes. The architect's job is to buy only the freshness the *decision* needs.

*"Would the business act differently on data that is 5 minutes old versus 5 hours old?"* If not, don't stream it.

In [ ]:
opts = pd.DataFrame({
    "option": ["Nightly batch", "Hourly micro-batch", "5-min micro-batch", "Streaming (seconds)", "Streaming (<200 ms)"],
    "latency_s": [12 * 3600, 3600, 300, 5, 0.2],
    "use_case": ["Regulatory reports,\nactuarial pricing", "Ops dashboards", "Claims queue\nprioritisation", "Telematics\nrisk scores", "Card / claim fraud\nblock-or-allow"],
})
rng_ = np.random.default_rng(0)
opts["cost_index"] = (1.0 + 18 / np.sqrt(opts.latency_s)) * rng_.uniform(0.95, 1.05, len(opts))     # simulated: cost ~ 1/sqrt(latency)
opts["complexity"] = [1, 2, 3, 7, 9]                                                                  # subjective 1-10 (infra + people + testing)
fig, ax = plt.subplots(figsize=(8.5, 4))
sc = ax.scatter(opts.latency_s, opts.cost_index, s=opts.complexity * 130, c=[TEAL, TEAL, GOLD, CORAL, CORAL], alpha=.85, edgecolor=INK)
for _, r_ in opts.iterrows():
    ax.annotate(f"{r_.option}\n{r_.use_case}", (r_.latency_s, r_.cost_index), textcoords="offset points", xytext=(0, 20), ha="center", fontsize=7.5)
ax.set_xscale("log"); ax.invert_xaxis(); ax.set_ylim(0, opts.cost_index.max() * 1.45)
ax.set_xlabel("data latency (seconds, log scale; fresher ->)"); ax.set_ylabel("relative run cost (simulated index)")
ax.set_title("Freshness vs cost vs complexity (bubble size = complexity)"); plt.tight_layout(); plt.show()

## 2.6 Architecture patterns: Lambda vs Kappa

| | **Lambda** | **Kappa** |
|---|---|---|
| Idea | Two paths: batch layer (accurate, slow) + speed layer (approximate, fast), merged at serving | **One** path: everything is a stream; "batch" = replaying the log from the start |
| Pros | Mature; batch gives authoritative recomputation; simple to reason about correctness | One codebase and one set of semantics; no reconciliation logic |
| Cons | **Two code paths** to keep in sync (a source of train/serve skew!); merge complexity | Needs a durable, replayable, long-retention log; reprocessing at scale needs engineering; stream engines must handle large state |
| Fits when | Heavy historical recompute, existing batch estate, strict reconciliation | Event-driven company, strong streaming skills, lakehouse tables that can be read as streams |

What we just built by hand was *Lambda in miniature*: the stream emitted fast approximate windows; the late-event side output was reconciled afterwards
to match the batch truth. Modern **lakehouse** engines (Delta/Iceberg/Hudi with Spark/Flink) blur the line - the same table can be written by a stream and read in batch - which is why many teams
drift toward a Kappa-flavoured design with batch *reads*.

### Your turn 2 - add a rule to the batch validator
The business tells you: *"any claim larger than the policy's `sum_insured` is impossible and must be quarantined."* Extend `validate` so such rows are rejected with reason `amount_exceeds_sum_insured`,
then test it on a copy of the landing data where you inflate 3 claim amounts.

In [ ]:
# TODO: write validate_v2(df) that does everything validate() does PLUS the new rule.
# HINT: call validate(df) first, then apply the extra rule on the 'good' part and move the offenders into 'rejected' with a reason.
def validate_v2(df):
    good, rejected = validate(df)
    return good, rejected     # <- add the sum_insured rule here

test = extract(raw_path).head(200).copy()
test.loc[test.index[[3, 7, 9]], "claim_amount"] = test.loc[test.index[[3, 7, 9]], "sum_insured"] * 1.5
# g2, r2 = validate_v2(test); print(len(r2), "rejected;", r2.reject_reason.tolist())

---
# Part 3 - Feature engineering (the architect's view)

A **feature** is a raw fact turned into something a model can use. Models do not see "a claim"; they see a vector of numbers. Choosing and computing those numbers is
usually worth more than choosing the algorithm - and it is where most production failures hide.

The common toolbox (we will use all of it on ACME claims):

| Technique | Example | Why |
|---|---|---|
| **Ratios / interactions** | `claim_amount / sum_insured` | A Rs 5 lakh claim means different things on a Rs 6 lakh vs Rs 60 lakh policy |
| **Recency / rolling counts** | days since last claim; claims in last 90 days | Behaviour over time; must be computed *as of* the event |
| **Cyclical encoding** | hour -> (sin, cos) | 23:00 and 01:00 are close; a raw integer says they are 22 apart |
| **Categorical encoding** | frequency / target encoding | Compress high-cardinality categories without exploding columns |
| **Text signals** | note length, keyword flags | Cheap structure from unstructured text (LLM embeddings come in Lab 3) |
| **Binning** | age bands | Robustness, interpretability, regulatory explainability |

**Architect's lens:** every feature is a *contract* between data engineering, data science and the serving system. Features should be
**named, versioned, documented, computed by one piece of code, and point-in-time correct.** The rest of this part shows what happens when they are not.

## 3.1 A feature registry: one definition, used everywhere
A **feature store** is, at its core, (1) a registry of feature definitions, (2) an *offline* store for training, (3) an *online* store for low-latency serving, and (4) machinery to
keep (2) and (3) consistent. We will build the beating heart of it - a registry of named functions - in a dozen lines. The key idea: **offline (a whole DataFrame) and online (one request)
call the very same function.**

In [ ]:
EXPECTED_RAW = ["claim_amount", "sum_insured", "premium", "customer_age", "tenure_years", "hour_reported", "claim_ts", "claim_ts_utc",
                "prev_claim_ts", "prior_claims", "claims_prior_90d", "notes", "loss_type", "region"]

def _days_since_last(d):
    gap = (pd.to_datetime(d["claim_ts"]) - pd.to_datetime(d["prev_claim_ts"])).dt.total_seconds() / 86400
    return gap.fillna(365).clip(upper=365)                # contract: "no previous claim" == 365 days (NOT 0, NOT NaN)

FEATURES = {   # name -> function(DataFrame) -> Series.  Pure functions of raw columns: no hidden state, no 'now()'.
    "amount_to_sum_insured": lambda d: d["claim_amount"] / d["sum_insured"],
    "amount_to_premium":     lambda d: np.log1p(d["claim_amount"] / d["premium"]),
    "log_amount":            lambda d: np.log1p(d["claim_amount"]),
    "hour_sin":              lambda d: np.sin(2 * np.pi * d["hour_reported"] / 24),
    "hour_cos":              lambda d: np.cos(2 * np.pi * d["hour_reported"] / 24),
    "is_night":              lambda d: ((d["hour_reported"] >= 22) | (d["hour_reported"] <= 4)).astype(int),
    "month_sin":             lambda d: np.sin(2 * np.pi * pd.to_datetime(d["claim_ts"]).dt.month / 12),
    "month_cos":             lambda d: np.cos(2 * np.pi * pd.to_datetime(d["claim_ts"]).dt.month / 12),
    "days_since_last_claim": _days_since_last,
    "quick_repeat":          lambda d: (_days_since_last(d) < 45).astype(int),
    "prior_claims":          lambda d: d["prior_claims"],
    "claims_prior_90d":      lambda d: d["claims_prior_90d"],
    "note_len":              lambda d: d["notes"].str.len(),
    "kw_no_witness":         lambda d: d["notes"].str.contains("No witnesses", case=False).astype(int),
    "kw_cash":               lambda d: d["notes"].str.contains("in cash", case=False).astype(int),
    "kw_no_fir":             lambda d: d["notes"].str.contains("FIR not filed", case=False).astype(int),
    "age_band":              lambda d: np.digitize(d["customer_age"], [25, 35, 50, 65]),
    "is_theft":              lambda d: (d["loss_type"] == "motor_theft").astype(int),
}
FEATURE_GROUPS = {
    "ratios":   ["amount_to_sum_insured", "amount_to_premium", "log_amount"],
    "time":     ["hour_sin", "hour_cos", "is_night", "month_sin", "month_cos"],
    "history":  ["days_since_last_claim", "quick_repeat", "prior_claims", "claims_prior_90d"],
    "text":     ["note_len", "kw_no_witness", "kw_cash", "kw_no_fir"],
    "segments": ["age_band", "is_theft"],
}
def build_matrix(df, names):
    """THE single code path for feature computation - called by training jobs AND by the serving API."""
    return pd.DataFrame({n: FEATURES[n](df) for n in names}, index=df.index)

show = build_matrix(claims, ["amount_to_sum_insured", "hour_sin", "hour_cos", "days_since_last_claim", "kw_no_witness", "age_band"])
print(f"{len(FEATURES)} registered features. Sample:"); show.head(4).round(3)

## 3.2 Does it pay off? A before/after experiment

We split **by time** (train on the first 70% of the calendar, test on the last 30%) - because in production the model predicts the *future*, so a random split would flatter us.
Metric: **ROC AUC** (1.0 = perfect ranking, 0.5 = coin flip), a natural fit for fraud triage where we rank claims for investigators.

Baseline = "raw columns as they arrive from the database" (with categories as integer codes). Then we add one feature group at a time.
We use a plain **logistic regression** deliberately: with an interpretable model the contribution of each feature group is visible.
(A tree ensemble recovers some interactions by itself - we check that too; the lesson survives, because trees still cannot see *text* or *history* you didn't give them.)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import GradientBoostingClassifier

claims = claims.sort_values("claim_ts").reset_index(drop=True)
cut = int(len(claims) * 0.7)
train_df, test_df = claims.iloc[:cut].copy(), claims.iloc[cut:].copy()
y_tr, y_te = train_df.is_fraud.values, test_df.is_fraud.values
print(f"train: {len(train_df)} claims up to {train_df.claim_date.max():%Y-%m-%d} | test: {len(test_df)} claims from {test_df.claim_date.min():%Y-%m-%d}")

RAW_NUM = ["claim_amount", "sum_insured", "premium", "customer_age", "tenure_years", "hour_reported"]
def raw_matrix(df):
    X = df[RAW_NUM].copy()
    X["loss_type_code"] = df.loss_type.map({k: i for i, k in enumerate(LOSS_TYPES)}); X["region_code"] = df.region.map({k: i for i, k in enumerate(REGIONS)})
    return X

class CategoricalEncoder:
    """Frequency + smoothed target encoding. NOTE: it has *state learned from training data* that must be shipped to serving."""
    def __init__(self, cols=("loss_type", "region"), smooth=30): self.cols, self.smooth = cols, smooth
    def fit(self, df, y):
        self.prior_ = y.mean(); self.freq_, self.te_ = {}, {}
        for c in self.cols:
            self.freq_[c] = df[c].value_counts(normalize=True).to_dict()
            g = pd.DataFrame({"c": df[c].values, "y": y}).groupby("c").y.agg(["sum", "count"])
            self.te_[c] = ((g["sum"] + self.smooth * self.prior_) / (g["count"] + self.smooth)).to_dict()
        return self
    def transform(self, df):
        out = {}
        for c in self.cols:
            out[f"{c}_freq"] = df[c].map(self.freq_[c]).fillna(0.0)
            out[f"{c}_te"] = df[c].map(self.te_[c]).fillna(self.prior_)          # unseen category -> prior
        return pd.DataFrame(out, index=df.index)

def fit_eval(Xtr, Xte, model="logreg"):
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0)) if model == "logreg" else GradientBoostingClassifier(n_estimators=120, learning_rate=.08, max_depth=3, random_state=SEED)
    m.fit(Xtr, y_tr)
    return m, roc_auc_score(y_te, m.predict_proba(Xte)[:, 1])

enc = CategoricalEncoder().fit(train_df, y_tr)
Xtr, Xte = raw_matrix(train_df), raw_matrix(test_df)
results = [("raw columns", fit_eval(Xtr, Xte)[1])]
for grp in ["ratios", "time", "history", "text", "segments"]:
    Xtr = pd.concat([Xtr, build_matrix(train_df, FEATURE_GROUPS[grp])], axis=1); Xte = pd.concat([Xte, build_matrix(test_df, FEATURE_GROUPS[grp])], axis=1)
    results.append((f"+ {grp}", fit_eval(Xtr, Xte)[1]))
Xtr = pd.concat([Xtr, enc.transform(train_df)], axis=1); Xte = pd.concat([Xte, enc.transform(test_df)], axis=1)
final_model, auc_final = fit_eval(Xtr, Xte); results.append(("+ encodings", auc_final))
FINAL_COLS = list(Xtr.columns)

abl = pd.DataFrame(results, columns=["feature set", "test_AUC"]).round(3); abl["lift_vs_raw"] = (abl.test_AUC - abl.test_AUC.iloc[0]).round(3)
print(abl.to_string(index=False))
_, gbm_raw = fit_eval(raw_matrix(train_df), raw_matrix(test_df), model="gbm"); _, gbm_all = fit_eval(Xtr, Xte, model="gbm")
print(f"\nSanity check with a gradient-boosted tree model: raw={gbm_raw:.3f}  engineered={gbm_all:.3f}")

fig, ax = plt.subplots(figsize=(8, 3.3))
cols = [TEAL] + [LIME] * (len(abl) - 2) + [GOLD]
ax.barh(abl["feature set"][::-1], abl.test_AUC[::-1], color=cols[::-1]); ax.set_xlim(0.5, abl.test_AUC.max() + .05)
for i, v in enumerate(abl.test_AUC[::-1]): ax.text(v + .003, i, f"{v:.3f}", va="center", fontsize=9)
ax.set_xlabel("AUC on held-out future claims"); ax.set_title("Feature engineering lift (cumulative)"); ax.grid(axis="y", alpha=0)
plt.tight_layout(); plt.show()

## 3.3 Silent killer #1: train/serve skew

The model scored a healthy AUC of ~0.88 in the notebook. In production it "feels worse" - but nothing crashed, no alert fired. This is **train/serve skew**: the feature values the model sees at serving time are
computed *differently* than at training time. It is almost never a dramatic bug. It is a timezone, a null-handling default, a rounding rule, a unit - written twice by two teams in two languages.

Here the data-science notebook computed `hour_reported` in local time and treated "no previous claim" as 365 days. The engineers re-implementing the logic in the serving API:
* took the hour from the **UTC** timestamp (our platform stores UTC),
* filled a missing previous-claim gap with **0** (a very natural "default"), and
* read the adjuster note from an online store whose column silently **truncates at 160 characters**, so the length and keyword features change.

Same model, same claims, only the feature code differs. Watch what happens to quality.

In [ ]:
def naive_online_features(df):
    """Re-implementation by 'the serving team'. Looks reasonable. Is wrong in two places."""
    d = df.copy()
    d["hour_reported"] = pd.to_datetime(d["claim_ts_utc"]).dt.hour                                    # BUG 1: UTC hour, not local
    d["notes"] = d["notes"].str.slice(0, 160)                                                         # BUG 3: online store column is VARCHAR(160)
    gap = (pd.to_datetime(d["claim_ts"]) - pd.to_datetime(d["prev_claim_ts"])).dt.total_seconds() / 86400
    out = build_matrix(d, [c for c in FINAL_COLS if c in FEATURES and c not in ("days_since_last_claim", "quick_repeat")])
    out["days_since_last_claim"] = gap.fillna(0)                                                      # BUG 2: null -> 0 (looks like a repeat claim!)
    out["quick_repeat"] = (gap.fillna(0) < 45).astype(int)
    d_raw = raw_matrix(d)
    return pd.concat([d_raw, out, enc.transform(d)], axis=1)[FINAL_COLS]

X_skew = naive_online_features(test_df)
X_shared = pd.concat([raw_matrix(test_df), build_matrix(test_df, [c for c in FINAL_COLS if c in FEATURES]), enc.transform(test_df)], axis=1)[FINAL_COLS]
auc_offline = roc_auc_score(y_te, final_model.predict_proba(Xte[FINAL_COLS])[:, 1])
auc_skew = roc_auc_score(y_te, final_model.predict_proba(X_skew)[:, 1])
auc_shared = roc_auc_score(y_te, final_model.predict_proba(X_shared)[:, 1])
print(f"Offline (training-time code)          AUC = {auc_offline:.3f}")
print(f"Online, re-implemented independently  AUC = {auc_skew:.3f}   <- silent degradation, no exceptions raised")
print(f"Online, via shared registry           AUC = {auc_shared:.3f}   <- identical to offline: max |diff| = {np.abs(X_shared.values - Xte[FINAL_COLS].values).max():.1e}")

# A cheap monitor that would have caught it: compare feature distributions train-vs-serve (here: standardised mean shift).
shift = ((X_skew.mean() - Xte[FINAL_COLS].mean()) / Xte[FINAL_COLS].std()).abs().sort_values(ascending=False).head(5).round(2)
print("\nTop skewed features (|mean shift| in std units) - the monitoring signal:\n", shift.to_string())

**What the architect takes away**
* Put feature logic in **one library/registry**, imported by the training pipeline *and* the serving service (or compute once and publish to both stores).
* **Log the features** the model actually saw at serving time and compare their distributions with training - skew shows up as drift long before anyone complains.
* Unit-test parity: for N sampled rows, offline result == online result (below, we do exactly that).

In [ ]:
# Parity test: score a handful of requests one-at-a-time, as an API would, and compare with the offline batch matrix.
def online_features(request: dict):
    """Serving path: one raw request (+ values looked up from the online store) -> feature vector. Same build_matrix() as training."""
    row = pd.DataFrame([request])
    return pd.concat([raw_matrix(row), build_matrix(row, [c for c in FINAL_COLS if c in FEATURES]), enc.transform(row)], axis=1)[FINAL_COLS]

sample = test_df.sample(25, random_state=1)
ok = all(np.allclose(online_features(rec).values[0], Xte.loc[idx, FINAL_COLS].values.astype(float), equal_nan=True) for idx, rec in zip(sample.index, sample.to_dict("records")))
print("Online == offline for 25 sampled requests:", ok)
req = sample.iloc[0].to_dict()
print(f"Fraud score for one live request (claim {req['claim_id']}): {final_model.predict_proba(online_features(req))[0, 1]:.3f}")

### Your turn 3 - extend the registry
Add a feature `is_weekend` (1 if the claim was filed on Saturday/Sunday) to `FEATURES`, then run a parity check that offline and online (single-row) produce the same value for 25 sampled rows.
*Why is this a good exercise?* Because the check you write is the same unit test that protects you from skew in production.

In [ ]:
# TODO 1: define is_weekend(d) -> Series of 0/1 using d["claim_ts"] (hint: pd.to_datetime(...).dt.dayofweek >= 5)
# TODO 2: register it:  FEATURES["is_weekend"] = is_weekend
# TODO 3: for 25 sampled rows compare  build_matrix(rows, ["is_weekend"])  with calling build_matrix on each single-row DataFrame.
pass

## 3.4 Silent killer #2: point-in-time leakage

**Leakage** means a feature smuggles in information that would *not have been available at prediction time*. It makes offline metrics look wonderful and production results collapse.
Classic forms: using the label to build a feature, aggregating over the full dataset (including the future), or joining a table that is updated *after* the event.

Example: *"customer's historical fraud rate."* It sounds like an excellent feature. Computed naively - `groupby(customer).mean(is_fraud)` over the whole table - it includes the very
claim we are predicting and every later claim. Computed **as of the claim time** it only sees the past. We train two models and test each both ways.

In [ ]:
def cust_fraud_rate_leaky(df_all):
    """Whole-table aggregate: includes the current row's label AND future rows. (What a hurried analyst writes.)"""
    return df_all.groupby("customer_id").is_fraud.transform("mean")

def cust_fraud_rate_asof(df_all, prior=0.10, k=2):
    """Point-in-time correct: expanding mean over STRICTLY EARLIER claims of the same customer (smoothed toward a prior)."""
    d = df_all.sort_values("claim_ts")
    cs = d.groupby("customer_id").is_fraud.cumsum() - d.is_fraud          # subtract self => only the past
    n = d.groupby("customer_id").cumcount()
    return ((cs + prior * k) / (n + k)).reindex(df_all.index)

base_cols = [c for c in FINAL_COLS]
leaky = cust_fraud_rate_leaky(claims); asof = cust_fraud_rate_asof(claims)
tr_idx, te_idx = train_df.index, test_df.index
Xtr_l, Xte_l = Xtr.assign(cust_fraud_rate=leaky.loc[tr_idx].values), Xte.assign(cust_fraud_rate=leaky.loc[te_idx].values)
Xte_a = Xte.assign(cust_fraud_rate=asof.loc[te_idx].values)
leaky_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xtr_l, y_tr)
auc_leaky_offline = roc_auc_score(y_te, leaky_model.predict_proba(Xte_l)[:, 1])
auc_leaky_prod = roc_auc_score(y_te, leaky_model.predict_proba(Xte_a)[:, 1])
asof_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xtr.assign(cust_fraud_rate=asof.loc[tr_idx].values), y_tr)
auc_asof = roc_auc_score(y_te, asof_model.predict_proba(Xte_a)[:, 1])
print(f"No such feature (honest baseline)                          AUC = {auc_final:.3f}")
print(f"LEAKY feature, evaluated offline (what the notebook says)  AUC = {auc_leaky_offline:.3f}   <- 'ship it!'")
print(f"LEAKY model, in production (feature as-of claim time)      AUC = {auc_leaky_prod:.3f}   <- reality")
print(f"Point-in-time correct feature, honest evaluation           AUC = {auc_asof:.3f}")

fig, ax = plt.subplots(figsize=(7.5, 3.1))
labs = ["honest baseline", "leaky: offline", "leaky: production", "point-in-time\ncorrect"]; vals = [auc_final, auc_leaky_offline, auc_leaky_prod, auc_asof]
ax.bar(labs, vals, color=[TEAL, GOLD, CORAL, LIME]); ax.set_ylim(0.5, 1.0)
for i, v in enumerate(vals): ax.text(i, v + .005, f"{v:.3f}", ha="center", fontweight="bold")
ax.set_ylabel("AUC"); ax.set_title("Leakage: too-good-to-be-true offline, disappointing live"); ax.grid(axis="x", alpha=0); plt.tight_layout(); plt.show()

**Rule of thumb for reviews:** if a model's offline accuracy is suspiciously good, suspect leakage *before* celebrating. In feature stores this is solved by **point-in-time joins** (a.k.a. "as-of" joins):
for each training example with timestamp *t*, fetch each feature's value as it was known at *t* - never later. That is why feature tables are stored with **event timestamps** and why
you must keep history (slowly-changing dimensions, snapshots) instead of overwriting rows in place.

---
# Part 4 - Data quality: trust, but verify (automatically)

*Garbage in, garbage out* is the oldest rule in computing, but AI raises the stakes: a model does not throw an error on bad data - it **learns it** and then makes confident, wrong decisions at scale.
Quality is not "clean the data once"; it is **continuous, automated testing of data**, the same way we unit-test code.

The six dimensions we will measure (a widely used vocabulary - you will hear variations of it from every vendor):

| Dimension | Question | ACME example |
|---|---|---|
| **Completeness** | Is anything missing that should be there? | `policy_id` null |
| **Uniqueness** | Is each real-world thing recorded once? | duplicate `claim_id` |
| **Validity** | Do values obey type, range and domain rules? | age = 230, amount < 0 |
| **Integrity** (referential) | Do references resolve? | claim points to a non-existent policy |
| **Timeliness** | Is it fresh enough, are partitions arriving? | no data for the last 12 days |
| **Consistency** | Do fields and sources agree with each other? | claim before policy start; amount above sum insured; unit change between feeds |

## 4.1 A reusable profiler: `quality_report`
First a quick, generic health check that works on *any* DataFrame - useful on day one with a new data source, before you know what the rules should be.

In [ ]:
def quality_report(df: pd.DataFrame, key=None) -> pd.DataFrame:
    """Column-level profile: completeness, cardinality, numeric range, plus a table-level duplicate-key count in .attrs."""
    rep = pd.DataFrame({"dtype": df.dtypes.astype(str), "non_null_pct": (100 * (1 - df.isna().mean())).round(1), "distinct": df.nunique()})
    num = df.select_dtypes("number")
    for stat in ("min", "median", "max"):
        rep[stat] = getattr(num, stat)().reindex(rep.index)
    if key is not None:
        rep.attrs["duplicate_keys"] = int(df.duplicated(key).sum())
    return rep

core = ["claim_id", "policy_id", "customer_age", "claim_amount", "sum_insured", "loss_type", "region", "claim_date"]
rep = quality_report(claims[core], key="claim_id")
print("duplicate claim_ids:", rep.attrs["duplicate_keys"]); rep

## 4.2 A tiny expectations engine (Great-Expectations / Soda / dbt-tests style)

A profile tells you what *is*; **expectations** declare what *must be true* and fail loudly when it is not. Each rule returns a structured result:
which dimension it belongs to, the share of rows (or partitions) that satisfy it (`score`), the threshold (`mostly`), and pass/fail. Hand-rolling it in ~70 lines demystifies the
commercial tools: they add scheduling, docs, alerting and integrations, but the core idea is this.

Design choices an architect should notice:
* Rules are **data** (a list of objects), not scattered `if` statements - they can be versioned, reviewed by data owners, and reused across datasets.
* A `mostly` threshold tolerates small imperfection (99% not-null) without hiding systemic failure.
* A missing column yields a **FAIL result**, not a crash - the guardrail must not itself be fragile.

In [ ]:
class Expectation:
    """One named rule. `fn(df, ctx)` returns (score in 0..1, detail). `ctx` carries reference data (dimension tables, as-of time, baselines)."""
    def __init__(self, name, dimension, fn, mostly=1.0, column=None):
        self.name, self.dimension, self.fn, self.mostly, self.column = name, dimension, fn, mostly, column
    def run(self, df, ctx):
        if self.column and self.column not in df.columns:
            score, detail = 0.0, "column missing (schema drift?)"
        else:
            try: score, detail = self.fn(df, ctx)
            except Exception as e: score, detail = 0.0, f"rule error: {type(e).__name__}"
        return {"rule": self.name, "dimension": self.dimension, "score": round(float(score), 4), "threshold": self.mostly, "passed": bool(score >= self.mostly), "detail": detail}

# --- rule factories -------------------------------------------------------------------------------------
def not_null(col, mostly=0.99):
    return Expectation(f"not_null[{col}]", "completeness", lambda d, c: (d[col].notna().mean(), f"{int(d[col].isna().sum())} nulls"), mostly, col)
def unique(col):
    return Expectation(f"unique[{col}]", "uniqueness", lambda d, c: (1 - d.duplicated(col).mean(), f"{int(d.duplicated(col).sum())} duplicate keys"), 1.0, col)
def between(col, lo, hi, mostly=0.99):
    return Expectation(f"between[{col}, {lo:g}..{hi:g}]", "validity", lambda d, c: ((ok := d[col].between(lo, hi)).mean(), f"{int((~ok).sum())} out of range"), mostly, col)
def in_set(col, allowed, mostly=0.999):
    return Expectation(f"in_set[{col}]", "validity", lambda d, c: ((ok := d[col].isin(allowed)).mean(), f"{int((~ok).sum())} unexpected values"), mostly, col)
def references(col, ref_name, ref_col, mostly=0.999):
    return Expectation(f"references[{col} -> {ref_name}.{ref_col}]", "integrity",
                       lambda d, c: ((ok := d[col].isin(c[ref_name][ref_col])).mean(), f"{int((~ok).sum())} orphan keys"), mostly, col)
def fresh(col, max_lag_days):
    def fn(d, c):
        lag = (c["as_of"] - pd.to_datetime(d[col]).max()).days
        return (1.0 if lag <= max_lag_days else max(0.0, 1 - (lag - max_lag_days) / 30)), f"newest record is {lag} days old (limit {max_lag_days})"
    return Expectation(f"fresh[{col} <= {max_lag_days}d]", "timeliness", fn, 1.0, col)
def no_gaps(col, last_n_days):
    def fn(d, c):
        days = pd.to_datetime(d[col]).dt.normalize(); window = pd.date_range(c["as_of"] - pd.Timedelta(days=last_n_days), c["as_of"] - pd.Timedelta(days=1))
        have = window.isin(days.unique()).mean(); return have, f"{have:.0%} of the last {last_n_days} days have data"
    return Expectation(f"no_gaps[{col}, last {last_n_days}d]", "timeliness", fn, 0.95, col)
def row_rule(name, dimension, predicate, mostly=0.999, column=None):
    """Cross-field consistency: predicate(df) -> boolean Series (True = row is OK)."""
    return Expectation(name, dimension, lambda d, c: ((ok := predicate(d)).mean(), f"{int((~ok).sum())} violating rows"), mostly, column)
def stable_median(col, by_month_of, tol=3.0):
    """Drift/unit check: each month's median must be within [1/tol, tol] x the trusted baseline median."""
    def fn(d, c):
        g = d.groupby(pd.to_datetime(d[by_month_of]).dt.to_period("M"))[col]
        med = g.median()[g.count() >= 30]; r_ = med / c["baseline_median"][col]        # ignore tiny partitions: medians of 3 rows are noise
        bad = r_[(r_ > tol) | (r_ < 1 / tol)]; return 1 - len(bad) / len(med), (f"drifting months: {[str(m) for m in bad.index]}" if len(bad) else "all months stable")
    return Expectation(f"stable_median[{col} by month]", "consistency", fn, 1.0, col)
def schema(expected_cols):
    def fn(d, c):
        have, want = set(d.columns), set(expected_cols); return len(have & want) / len(have | want), f"missing={sorted(want - have)} unexpected={sorted(have - want)}"
    return Expectation("schema_matches_contract", "consistency", fn, 1.0)

def run_suite(df, suite, ctx) -> pd.DataFrame:
    return pd.DataFrame([e.run(df, ctx) for e in suite])

# --- the ACME claims suite --------------------------------------------------------------------------------
CLAIM_COLUMNS = list(claims.columns)
def claims_suite():
    return [
        schema(CLAIM_COLUMNS),
        not_null("policy_id", 0.999), not_null("claim_amount", 1.0), not_null("customer_age", 0.98),
        unique("claim_id"),
        between("claim_amount", 1_000, 5_000_000, 0.995), between("customer_age", 18, 100, 0.999),
        in_set("loss_type", LOSS_TYPES), in_set("region", REGIONS),
        references("policy_id", "policies", "policy_id"),
        fresh("claim_date", max_lag_days=3), no_gaps("claim_date", 14),
        row_rule("claim_date >= policy_start", "consistency", lambda d: d.claim_date >= d.policy_start, 0.999, "policy_start"),
        row_rule("claim_amount <= sum_insured", "consistency", lambda d: d.claim_amount <= d.sum_insured, 0.999, "claim_amount"),
        stable_median("claim_amount", by_month_of="claim_date"),
    ]

CTX = {"policies": policies, "as_of": claims.claim_date.max() + pd.Timedelta(days=1), "baseline_median": {"claim_amount": claims.claim_amount.median()}}
clean_results = run_suite(claims, claims_suite(), CTX)
print(f"CLEAN data: {clean_results.passed.sum()}/{len(clean_results)} rules pass")
clean_results[["rule", "dimension", "score", "passed", "detail"]]

## 4.3 Break it on purpose: realistic defects

A monitoring system you have never seen fail is a system you cannot trust. We copy the clean table and inject the defects that actually occur in enterprise feeds:

| Defect | Real-world cause |
|---|---|
| Stale partition | An upstream job failed; the last 12 days never arrived |
| Duplicates | Retries / double-delivery (Part 2!) |
| Nulls | A form field became optional in a new app release |
| Impossible values | Typos, test data in production, sign errors |
| Unit mismatch | One feed switches from rupees to *thousands* of rupees |
| Orphan keys | Claims loaded before the policy master caught up |
| Cross-field violations | Back-dated claims; amounts above cover |
| Schema drift | Upstream renames `region` to `state` without telling anyone |

In [ ]:
def inject_defects(df, seed=99):
    r = np.random.default_rng(seed); d = df.copy(); log = {}
    cutoff = d.claim_date.max() - pd.Timedelta(days=12)
    log["stale partition: rows removed"] = int((d.claim_date > cutoff).sum()); d = d[d.claim_date <= cutoff].copy()
    dup = d.sample(40, random_state=1); d = pd.concat([d, dup], ignore_index=True); log["duplicate rows"] = 40
    idx = r.choice(len(d), int(.015 * len(d)), replace=False); d.loc[idx, "policy_id"] = None; log["null policy_id"] = len(idx)
    idx = r.choice(len(d), int(.08 * len(d)), replace=False); d.loc[idx, "customer_age"] = np.nan; log["null customer_age"] = len(idx)
    idx = r.choice(len(d), 25, replace=False); d.loc[idx, "claim_amount"] = -d.loc[idx, "claim_amount"]; log["negative amounts"] = 25
    idx = r.choice(len(d), 12, replace=False); d.loc[idx, "customer_age"] = 230; log["age = 230"] = 12
    m = d.claim_date.dt.strftime("%Y-%m") == "2024-09"; d.loc[m, "claim_amount"] /= 1000; log["unit mismatch (Sep-2024 in thousands)"] = int(m.sum())
    idx = r.choice(len(d), 30, replace=False); d.loc[idx, "policy_id"] = [f"P9{i:05d}" for i in range(30)]; log["orphan policy_id"] = 30
    idx = r.choice(len(d), 20, replace=False); d.loc[idx, "claim_date"] = d.loc[idx, "policy_start"] - pd.Timedelta(days=30); log["claim before policy start"] = 20
    idx = r.choice(len(d), 25, replace=False); d.loc[idx, "claim_amount"] = d.loc[idx, "sum_insured"] * 1.8; log["amount above sum insured"] = 25
    d = d.rename(columns={"region": "state"}); log["schema drift: region -> state"] = 1
    return d, log

dirty, defect_log = inject_defects(claims)
print(pd.Series(defect_log, name="injected").to_string())
print(f"\noriginal untouched: {claims.claim_amount.min() > 0 and 'region' in claims.columns} | dirty shape: {dirty.shape}")

### Did the rules catch them?
Run the *same* suite on the dirty copy and compare. A good suite is judged by **detection** (every injected defect trips at least one rule) and by **diagnosis** (the `detail` column tells the on-call engineer what to look at).

In [ ]:
dirty_results = run_suite(dirty, claims_suite(), CTX)
cmp_q = clean_results[["rule", "dimension", "passed"]].merge(dirty_results[["rule", "score", "passed", "detail"]], on="rule", suffixes=("_clean", "_dirty"))
cmp_q["verdict"] = np.where(cmp_q.passed_dirty, "PASS", "FAIL")
print(f"DIRTY data: {dirty_results.passed.sum()}/{len(dirty_results)} rules pass  ->  {(~dirty_results.passed).sum()} failures detected\n")
print(cmp_q[["rule", "score", "verdict", "detail"]].to_string(index=False))

### The data-quality scorecard
Executives do not read rule lists; they read a scorecard. We roll rule scores up to **dimension level** and show clean vs dirty. In production this is the dashboard that gates a pipeline:
*"if overall quality < 95% or any critical rule fails, stop the downstream training job and page the data owner"* - a **quality gate**.

In [ ]:
def scorecard(results):
    return results.groupby("dimension").score.mean().mul(100)

sc_clean, sc_dirty = scorecard(clean_results), scorecard(dirty_results)
order = ["completeness", "uniqueness", "validity", "integrity", "timeliness", "consistency"]
fig, ax = plt.subplots(figsize=(8, 3.8))
y = np.arange(len(order)); h = .36
ax.barh(y + h / 2, sc_clean.reindex(order), height=h, color=TEAL, label="clean feed"); ax.barh(y - h / 2, sc_dirty.reindex(order), height=h, color=CORAL, label="dirty feed")
ax.axvline(95, color=INK, ls="--", lw=1); ax.text(95.5, len(order) - .45, "quality gate 95%", fontsize=8)
for i, k in enumerate(order): ax.text(sc_dirty[k] + 1, i - h / 2, f"{sc_dirty[k]:.0f}", va="center", fontsize=8, color=CORAL)
ax.set_yticks(y); ax.set_yticklabels(order); ax.invert_yaxis(); ax.set_xlim(0, 112); ax.set_xlabel("dimension score (mean rule score, %)")
ax.set_title("Data-quality scorecard: ACME claims feed"); ax.legend(loc="lower left", fontsize=8); ax.grid(axis="y", alpha=0)
plt.tight_layout(); fig.savefig(ART / "quality_scorecard.png", dpi=120); plt.show()
gate = "OPEN (train)" if (dirty_results.passed.all()) else "CLOSED - downstream training blocked, owner notified"
print("Quality gate for dirty feed:", gate)

## 4.4 What does bad data actually cost the model?

Rules are an investment; here is the return. We degrade the **training data** in two ways that mirror real defects and measure AUC on clean held-out claims:

1. **Label noise** - some fraud flags are wrong (missed fraud, wrongly flagged customers).
2. **Missingness** - a growing share of feature values are blank (imputed with the median, the lazy default).

Each point averages 3 random draws to smooth the curve.

In [ ]:
from sklearn.impute import SimpleImputer
def auc_with_noise(label_noise=0.0, missing=0.0, reps=3):
    out = []
    for rep_ in range(reps):
        r = np.random.default_rng(100 + rep_)
        y_noisy = np.where(r.random(len(y_tr)) < label_noise, 1 - y_tr, y_tr)
        A, B = Xtr[FINAL_COLS].astype(float).copy(), Xte[FINAL_COLS].astype(float).copy()
        if missing > 0:
            A = A.mask(r.random(A.shape) < missing); B = B.mask(r.random(B.shape) < missing)
        m = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=2000)).fit(A, y_noisy)
        out.append(roc_auc_score(y_te, m.predict_proba(B)[:, 1]))
    return np.mean(out)

noise_grid = [0, .02, .05, .10, .20, .30]; miss_grid = [0, .1, .2, .3, .5, .7]
auc_noise = [auc_with_noise(label_noise=p) for p in noise_grid]
auc_miss = [auc_with_noise(missing=p) for p in miss_grid]

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
ax[0].plot([p * 100 for p in noise_grid], auc_noise, "o-", color=CORAL, lw=2); ax[0].set_xlabel("% training labels flipped"); ax[0].set_ylabel("AUC on clean test data"); ax[0].set_title("Label noise")
ax[1].plot([p * 100 for p in miss_grid], auc_miss, "o-", color=GOLD, lw=2); ax[1].set_xlabel("% of feature values missing (mean-imputed)"); ax[1].set_title("Missing data")
for a in ax: a.axhline(auc_noise[0], color=TEAL, ls=":", lw=1)
plt.suptitle("The cost of bad data: model quality decays smoothly - which is why nobody notices", fontweight="bold", y=1.03); plt.tight_layout(); plt.show()
print(pd.DataFrame({"label_noise_%": [int(p * 100) for p in noise_grid], "AUC": np.round(auc_noise, 3), "missing_%": [int(p * 100) for p in miss_grid], "AUC ": np.round(auc_miss, 3)}).to_string(index=False))

**The business translation.** If investigators can review the top 5% of claims, a drop of a few AUC points is real leakage: fraud that would have been caught is paid out. The decay is *gradual*,
so without an automated check you will not see a cliff - you will see a quarter of quietly worse loss ratios. This is the argument for funding data quality as a **product feature**, not a cleanup project.

### Your turn 4 - add a validity rule
Phone numbers must look like `+91 ` followed by 10 digits. Write a rule using `row_rule` (dimension `validity`, tolerance 99%) and test it on a copy of `claims` where you corrupt 5% of the phone numbers.

In [ ]:
# TODO: build phone_rule = row_rule(name, "validity", predicate, mostly=0.99, column="phone")
# HINT: predicate = lambda d: d.phone.str.fullmatch(r"\+91 \d{10}")  (returns True when the row is OK)
bad_phones = claims.copy()
bad_phones.loc[bad_phones.sample(frac=0.05, random_state=0).index, "phone"] = "98-765"
phone_rule = None   # <- replace
# print(phone_rule.run(claims, CTX)); print(phone_rule.run(bad_phones, CTX))

---
# Part 5 - Labeling: where ground truth comes from

Supervised AI learns from examples *somebody labelled*. At ACME, the label might be "this claim was referred to the Special Investigations Unit (SIU)" or "adjuster tagged the document as a medical invoice".
Labels are **opinions captured at a cost** - they are noisy, expensive, slow, and often the real bottleneck of a project.

Three architect questions:
1. **How noisy are our labels?** (Measure *inter-annotator agreement*, don't assume.)
2. **How do we combine disagreeing labellers?** (Majority vote? Weighted by reliability?)
3. **How do we spend a limited labelling budget?** (Label the *informative* items first: active learning.)

For this part we use a synthetic stand-in for "claim triage" features: 4,000 items, 25% positive, where we secretly know the truth - so we can grade the labellers, which you can never do in real life
(that is why real programmes keep a small **gold set** labelled by senior experts and used only for auditing).

In [ ]:
from sklearn.datasets import make_classification
from sklearn.metrics import cohen_kappa_score, roc_auc_score
from sklearn.linear_model import LogisticRegression

Xl, yl = make_classification(n_samples=4000, n_features=20, n_informative=6, n_redundant=4, weights=[.75, .25], class_sep=1.0, flip_y=0, random_state=3)
Xpool, ypool, Xtest, ytest = Xl[:2500], yl[:2500], Xl[2500:], yl[2500:]
N_LAB = 300                                              # items we pay people to label
ANN_ACC = {"Asha (senior)": 0.92, "Bilal": 0.85, "Chen (new joiner)": 0.75}      # true accuracies (hidden in real life)

def annotate(y_true, accs, rng):
    """Each annotator independently reports the true label with prob = accuracy, else flips it."""
    return np.column_stack([np.where(rng.random(len(y_true)) < a, y_true, 1 - y_true) for a in accs])

A = annotate(ypool[:N_LAB], ANN_ACC.values(), np.random.default_rng(0))
labels_df = pd.DataFrame(A, columns=ANN_ACC.keys())
print(f"{N_LAB} items labelled by 3 annotators. Base rate of 'refer to SIU' = {ypool[:N_LAB].mean():.0%}")
labels_df.head(5).assign(truth=ypool[:5])

## 5.1 Inter-annotator agreement
**Cohen's kappa** compares observed agreement with the agreement expected by chance (two people who label "no" 75% of the time will agree often by luck).
* kappa = 1: perfect, 0: no better than chance. Rough guide (Landis-Koch): <0.2 slight, 0.2-0.4 fair, 0.4-0.6 moderate, 0.6-0.8 substantial, >0.8 almost perfect.
* **Fleiss' kappa** generalises to more than two raters.

Why an architect cares: **a model cannot be more consistent than the people who labelled its training data.** Low kappa means the *task definition or guidelines* are ambiguous - and
no amount of model tuning fixes that. Fix the labelling instructions first.

In [ ]:
names = list(ANN_ACC)
K = np.array([[cohen_kappa_score(labels_df[a], labels_df[b]) if a != b else 1.0 for b in names] for a in names])

def fleiss_kappa(ratings, n_cat=2):
    """ratings: (items x raters) array of category ids."""
    n_items, n_raters = ratings.shape
    counts = np.stack([(ratings == c).sum(1) for c in range(n_cat)], axis=1)            # items x categories
    p_j = counts.sum(0) / (n_items * n_raters)                                          # category prevalence
    P_i = ((counts ** 2).sum(1) - n_raters) / (n_raters * (n_raters - 1))               # per-item agreement
    P_bar, P_e = P_i.mean(), (p_j ** 2).sum()
    return (P_bar - P_e) / (1 - P_e)

fk = fleiss_kappa(A)
fig, ax = plt.subplots(figsize=(4.8, 3.8))
im = ax.imshow(K, cmap="YlGnBu", vmin=0, vmax=1); ax.set_xticks(range(3)); ax.set_yticks(range(3)); ax.set_xticklabels([n.split(" ")[0] for n in names]); ax.set_yticklabels([n.split(" ")[0] for n in names]); ax.grid(False)
for i in range(3):
    for j in range(3): ax.text(j, i, f"{K[i, j]:.2f}", ha="center", va="center", color="white" if K[i, j] > .55 else INK, fontweight="bold")
ax.set_title(f"Pairwise Cohen's kappa (Fleiss = {fk:.2f})"); plt.tight_layout(); plt.show()
band = "almost perfect" if fk > .8 else "substantial" if fk > .6 else "moderate" if fk > .4 else "fair" if fk > .2 else "slight"
print(f"Fleiss' kappa = {fk:.2f}  -> '{band}' agreement (Landis-Koch bands). "
      f"Unanimous on {(A.min(1) == A.max(1)).mean():.0%} of items; the other {(A.min(1) != A.max(1)).mean():.0%} are disputed.")

## 5.2 Aggregating labels: majority vote vs reliability-weighted vote
**Majority vote** treats everyone equally. If we can estimate each annotator's reliability, a **weighted vote** lets the careful annotators count for more.
Without gold labels we can still estimate reliability from the crowd itself - an EM-style idea in the Dawid-Skene family:
1. start with the majority vote as a provisional truth, 2. estimate each annotator's accuracy against it, 3. re-vote with weights `log(acc / (1-acc))`, 4. repeat.

Caveat: with only 300 items and 3 raters the reliability estimates are themselves noisy (a few points either way) - so weighted voting usually helps only modestly here, and more when raters differ a lot or there are more of them.

(Intuition for the weight: an annotator who is right 90% of the time provides far more evidence per vote than one who is right 70% of the time.)

In [ ]:
def majority_vote(A): return (A.mean(1) > .5).astype(int)

def weighted_vote(A, iters=10):
    consensus = majority_vote(A)
    for _ in range(iters):
        acc = np.clip((A == consensus[:, None]).mean(0), .51, .99)           # reliability vs current consensus
        w = np.log(acc / (1 - acc))
        consensus = ((2 * A - 1) @ w > 0).astype(int)
    return consensus, acc

mv = majority_vote(A); wv, est_acc = weighted_vote(A)
truth = ypool[:N_LAB]
print(pd.DataFrame({"annotator": names, "true_accuracy": list(ANN_ACC.values()), "estimated_from_crowd_only": est_acc.round(2),
                    "actual_accuracy_vs_truth": (A == truth[:, None]).mean(0).round(2)}).to_string(index=False))
print(f"\nLabel quality vs hidden truth:  best single = {(A == truth[:, None]).mean(0).max():.1%} | majority vote = {(mv == truth).mean():.1%} | weighted vote = {(wv == truth).mean():.1%}")

## 5.3 Does label quality change the model?
We train the same classifier on 300 labels from different sources and score it on a clean held-out set. To avoid reading too much into one lucky draw, we repeat the annotation 25 times.
Compare: the weakest annotator alone, the strongest alone, majority, weighted, and the (unobtainable) gold standard.

In [ ]:
def test_auc(y_train):
    m = LogisticRegression(max_iter=1000).fit(Xpool[:N_LAB], y_train)
    return roc_auc_score(ytest, m.predict_proba(Xtest)[:, 1])

rows = []
for rep_ in range(25):
    Ar = annotate(truth, ANN_ACC.values(), np.random.default_rng(1000 + rep_)); w_, _ = weighted_vote(Ar)
    rows.append({"worst annotator": test_auc(Ar[:, 2]), "best annotator": test_auc(Ar[:, 0]), "majority vote": test_auc(majority_vote(Ar)),
                 "weighted vote": test_auc(w_), "gold (clean)": test_auc(truth)})
res = pd.DataFrame(rows); summ = res.agg(["mean", "std"]).T.round(3)
print(summ.to_string())

fig, ax = plt.subplots(figsize=(7.5, 3.2))
ax.bar(summ.index, summ["mean"], yerr=summ["std"], color=[CORAL, GOLD, TEAL, TEAL, LIME], capsize=4); ax.set_ylim(summ["mean"].min() - .08, summ["mean"].max() + .03)
ax.set_ylabel("test AUC (mean +/- sd, 25 repeats)"); ax.set_title("Same model, same 300 items, different label sources"); ax.grid(axis="x", alpha=0); plt.setp(ax.get_xticklabels(), rotation=12); plt.tight_layout(); plt.show()

**Reading the chart:** aggregation recovers a good share of the gap between a noisy annotator and gold, at the price of paying 3x per item. The right trade-off depends on the cost of an extra label versus the cost of a wrong model decision:
* high-stakes, low-volume (medical, claims > Rs 10 lakh): multiple expert labels + adjudication;
* low-stakes, high-volume: one label, plus a small gold set to monitor annotator quality.

### Your turn 5 - route the disputes
Instead of paying 3 annotators for *every* item, a cheaper workflow sends items to a senior **adjudicator** only when the 3 annotators disagree. Compute (a) the share of items that would go to the adjudicator,
(b) the accuracy of the majority vote on *unanimous* items vs *disputed* items.

In [ ]:
# TODO: unanimous = boolean array (all three annotators equal); then compute the share disputed and majority-vote accuracy in each group.
# HINT: A.min(axis=1) == A.max(axis=1) is True when all annotators agree.
unanimous = None   # <- replace
# print(...)

## 5.4 Active learning: label the *right* items
Labelling items at random wastes money on the easy ones the model already understands. **Uncertainty sampling** asks humans to label the items the current model is *least sure about* (predicted probability near 0.5).
Loop: train -> score unlabelled pool -> send the 10 most uncertain items to annotators -> retrain.

We compare against random sampling, starting from 20 labelled items (5 positives, 15 negatives) and adding 10 per round, averaged over 10 random starts. Labels here come from the truth oracle to isolate the sampling effect.

In [ ]:
def active_run(strategy, seed, rounds=20, batch=10, init=20):
    r = np.random.default_rng(seed)
    pos, neg = np.where(ypool == 1)[0], np.where(ypool == 0)[0]
    labelled = list(r.choice(pos, 5, replace=False)) + list(r.choice(neg, init - 5, replace=False)); curve = []
    for _ in range(rounds + 1):
        m = LogisticRegression(max_iter=1000).fit(Xpool[labelled], ypool[labelled])
        curve.append(roc_auc_score(ytest, m.predict_proba(Xtest)[:, 1]))
        unl = np.setdiff1d(np.arange(len(ypool)), labelled)
        if strategy == "random": pick = r.choice(unl, batch, replace=False)
        else: pick = unl[np.argsort(np.abs(m.predict_proba(Xpool[unl])[:, 1] - .5))[:batch]]        # most uncertain first
        labelled += list(pick)
    return curve

n_labels = 20 + 10 * np.arange(21)
curves = {s: np.mean([active_run(s, i) for i in range(10)], axis=0) for s in ("random", "uncertainty")}
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.plot(n_labels, curves["random"], "o-", color=GOLD, lw=2, label="random sampling"); ax.plot(n_labels, curves["uncertainty"], "o-", color=TEAL, lw=2, label="uncertainty sampling")
ax.set_xlabel("number of labelled items"); ax.set_ylabel("test AUC"); ax.set_title("Active learning: same budget, better model"); ax.legend(); plt.tight_layout(); plt.show()
target = round(curves["random"][-1] - 0.01, 3)          # 'good enough' = within 0.01 AUC of what random sampling reaches with the full budget
need = {s: int(n_labels[np.argmax(c >= target)]) if (c >= target).any() else None for s, c in curves.items()}
print(f"Labels needed to reach AUC {target}: random = {need['random']}, uncertainty = {need['uncertainty']}")
if need["random"] and need["uncertainty"]:
    print(f"=> {1 - need['uncertainty'] / need['random']:.0%} fewer labels. At Rs 40/label that is Rs {(need['random'] - need['uncertainty']) * 40:,} saved on this toy task - real projects label millions.")

### Beyond humans-only labelling (what to know, at architect level)
* **Weak supervision (e.g. Snorkel-style labelling functions):** encode heuristics ("note contains 'no witnesses' AND amount > 40% of cover -> probably suspicious") as many noisy voters, then *learn* their accuracies and combine them - exactly the weighted-vote idea above, at scale. Great for bootstrapping.
* **LLM-assisted labelling:** an LLM pre-labels (or drafts a label plus rationale) and humans verify or correct. This can cut cost sharply, but treat the LLM as **one more annotator with its own error profile**: measure its kappa against a gold set, watch for systematic bias,
  keep humans on the disputed/high-stakes slice, and **never evaluate a model on labels produced by the same LLM** (circular validation).
* **Governance of labelling:** version the labelling guidelines, record *who* labelled *what* and *when* (lineage), protect PII shown to external annotators, and keep a frozen gold set for regression testing.
* **Feedback loops:** production decisions generate labels too (investigator outcomes). Beware **selection bias** - you only learn outcomes for claims you chose to investigate.

---
# Part 6 - Governance: ownership, classification, privacy, access

Governance sounds like paperwork until the day a regulator, an auditor or a journalist asks **"what data did this model use, where did it come from, who approved it, and why did a call-centre agent see that field?"**
For AI the questions sharpen: *Was personal data used to train? For what purpose was it collected? Can we delete one person's data and prove it left the features and the model?*

Four building blocks we will implement as code, because **governance that lives only in a PDF is not enforced**:

1. A **catalogue** (what exists, who owns it, how sensitive it is, where it came from, how long we keep it).
2. **PII detection and de-identification** (find it, mask it, pseudonymise it).
3. **Anonymity checks** (k-anonymity: can records be re-identified from innocuous fields?).
4. **Policy-based access control** (who may see which columns and rows).

## 6.1 A data catalogue with machine-checkable policy
Each dataset gets an entry. The important part is not the table - it is that a **linter** can validate every entry against company policy automatically (in real platforms this is what catalogues such as Unity Catalog, Purview, Collibra or DataHub enable via tags and policies).

In [ ]:
CLASSES = ["public", "internal", "confidential", "pii"]          # increasing sensitivity
catalogue = pd.DataFrame([
    # dataset,                     owner,              classification, upstream,                          retention_days, purpose,                         lawful_basis
    ("raw.claims_fnol_json",       "Claims Ops",       "pii",          [],                                 2555,  "claims handling",                "contract"),
    ("raw.adjuster_notes",         "Claims Ops",       "pii",          [],                                 2555,  "claims handling",                "contract"),
    ("raw.telematics_events",      None,               "pii",          [],                                 None,  "usage-based pricing",            None),
    ("curated.claims",             "Data Platform",    "pii",          ["raw.claims_fnol_json", "raw.adjuster_notes"], 2555, "claims handling, fraud prevention", "contract; legitimate interest"),
    ("features.fraud_v1",          "Fraud Analytics",  "confidential", ["curated.claims"],                 730,   "fraud prevention",               "legitimate interest"),
    ("reports.claims_kpi",         "Finance",          "internal",     ["curated.claims"],                 1825,  "management reporting",           "legitimate interest"),
    ("ref.loss_type_codes",        "Data Platform",    "public",       [],                                 3650,  "reference data",                 "n/a"),
], columns=["dataset", "owner", "classification", "upstream", "retention_days", "purpose", "lawful_basis"])

def lint_catalogue(cat, max_pii_retention=2555):
    issues = []
    known = set(cat.dataset)
    for r in cat.itertuples():
        if not r.owner: issues.append((r.dataset, "no accountable owner"))
        if r.classification not in CLASSES: issues.append((r.dataset, f"unknown classification {r.classification}"))
        if r.classification == "pii":
            if r.retention_days is None or pd.isna(r.retention_days): issues.append((r.dataset, "PII without a retention period (storage limitation)"))
            elif r.retention_days > max_pii_retention: issues.append((r.dataset, f"PII retention {r.retention_days}d exceeds policy {max_pii_retention}d"))
            if not r.lawful_basis or pd.isna(r.lawful_basis): issues.append((r.dataset, "PII without a documented lawful basis/purpose"))
        for u in r.upstream:
            if u not in known: issues.append((r.dataset, f"lineage points to unknown dataset {u}"))
    return pd.DataFrame(issues, columns=["dataset", "issue"])

print("Catalogue linter findings:"); print(lint_catalogue(catalogue).to_string(index=False))

def show_lineage(cat, ds, depth=0):
    print("  " * depth + ("└─ " if depth else "") + ds + f"   [{cat.set_index('dataset').loc[ds, 'classification']}]")
    for u in cat.set_index("dataset").loc[ds, "upstream"]: show_lineage(cat, u, depth + 1)
print("\nLineage of the fraud feature table (upstream sources):"); show_lineage(catalogue, "features.fraud_v1")

Notice three things an auditor would value: (1) a dataset with **no owner** is flagged, (2) PII with **no retention period / lawful basis** is flagged, (3) **lineage** lets you answer *"if `raw.adjuster_notes` has a defect or a deletion request, which downstream datasets and models are affected?"* by walking the graph the other way.

**Purpose limitation** deserves a call-out for AI: data collected "for claims handling" is not automatically fair game for training a marketing model. The catalogue's `purpose` column is where that decision is recorded and later enforced.

## 6.2 Finding PII automatically
Adjuster notes are free text, and free text is where PII hides: phone numbers typed into comments, card numbers pasted by customers, IDs in email bodies. A first line of defence is pattern-based detection.

Pattern alone gives **false positives** (a 16-digit order number looks like a card), so for card numbers we add the **Luhn checksum** - the check-digit algorithm real card numbers satisfy.
We also resolve overlaps (first match wins by priority).

In [ ]:
def luhn_ok(digits: str) -> bool:
    ds = [int(c) for c in digits][::-1]
    return sum(d if i % 2 == 0 else (d * 2 - 9 if d * 2 > 9 else d * 2) for i, d in enumerate(ds)) % 10 == 0

PII_PATTERNS = {           # priority order matters: first accepted match claims the characters
    "card":        r"\b(?:\d[ -]?){12,18}\d\b",                                   # validated with Luhn below
    "national_id": r"(?<!\d\s)\b\d{4}\s\d{4}\s\d{4}\b(?!\s\d)",                                    # 12 digits in 4-4-4 groups (Aadhaar-like *shape* only; no checksum validation here)
    "email":       r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b",
    "phone":       r"(?:\+91[\s-]?)?\b[6-9]\d{4}[\s-]?\d{5}\b",
}

def detect_pii(text):
    found, taken = [], []
    for kind, pat in PII_PATTERNS.items():
        for m in re.finditer(pat, text):
            if any(m.start() < e and s < m.end() for s, e in taken): continue      # overlaps something already claimed
            if kind == "card" and not (13 <= len(re.sub(r"\D", "", m.group())) <= 19 and luhn_ok(re.sub(r"\D", "", m.group()))): continue
            found.append({"type": kind, "value": m.group(), "span": (m.start(), m.end())}); taken.append((m.start(), m.end()))
    return sorted(found, key=lambda f: f["span"])

tricky = ("Customer Priya called from +91 98765 43210, email priya.iyer@example.com. Paid with card 4111 1111 1111 1111. "
          "Order ref 1234 5678 9012 3456 is NOT a card. National id on file: 2345 6789 0123. Policy number 6123456789 (not a phone!).")
for f in detect_pii(tricky): print(f"{f['type']:<12} {f['value']}")
print("\nNote the last line: a 10-digit policy number starting with 6 is flagged as a phone. Regex = recall, not precision -> production tools add context, dictionaries and ML-based entity recognition (NER).")

In [ ]:
# Scan every adjuster note in the platform: how widespread is PII in 'just text'?
scan = Counter()
for t in claims.notes: scan.update({f["type"] for f in detect_pii(t)})
print({k: f"{v / len(claims):.0%} of notes" for k, v in scan.items()})
print("\nEvery note carries contact details => if these notes are sent to an external LLM or analytics tool unredacted, you have just disclosed customer PII to a third party.")

## 6.3 De-identify: mask, redact, pseudonymise
| Technique | What it does | Reversible? | Use for |
|---|---|---|---|
| **Redact / mask** | Replace with `[PHONE]` or `XXXXXX1234` | No | Free text going to analytics/LLMs |
| **Pseudonymise (tokenise)** | Replace with a stable token derived from a *secret* | Only with the secret/vault | Join keys across datasets without exposing identity |
| **Generalise / aggregate** | Age 34 -> 30-39; pincode 400053 -> 400 | No | Statistical releases (see k-anonymity next) |
| **Drop** | Remove the column | No | Data minimisation - the best control is not holding the data |

**Pseudonymisation is not anonymisation**: under regimes like GDPR, pseudonymised data is still personal data, because a key exists that could re-link it. And a bare, **unsalted hash of a low-entropy value is not protection**: phone numbers have only ~10^10 possibilities.
We use **HMAC with a secret key** (kept in a vault/KMS - never in code; the constant below is for the demo only).

In [ ]:
DEMO_SECRET = b"demo-only-secret-rotate-and-store-in-a-vault"
def pseudonymise(value: str, kind="id") -> str:
    return f"{kind.upper()}_" + hmac.new(DEMO_SECRET, value.encode(), hashlib.sha256).hexdigest()[:10]

def redact(text, mode="mask"):
    """Replace every detected PII span. mode='mask' -> [TYPE]; mode='pseudo' -> stable token."""
    out, last = [], 0
    for f in detect_pii(text):
        s, e = f["span"]; out.append(text[last:s]); out.append(f"[{f['type'].upper()}]" if mode == "mask" else pseudonymise(f["value"], f["type"])); last = e
    out.append(text[last:]); return "".join(out)

sample_note = claims.notes.iloc[11]
print("ORIGINAL :", sample_note); print("MASKED   :", redact(sample_note)); print("PSEUDONYM:", redact(sample_note, "pseudo"))
print("\nSame phone on two different claims -> same token (joinable, but not readable):",
      pseudonymise("+91 9876543210", "phone") == pseudonymise("+91 9876543210", "phone"))

# why unsalted hashes of phone numbers are a bad idea: a 'dictionary attack' on a leaked hash column
target_phone = "9876512345"; leaked_hash = hashlib.sha256(target_phone.encode()).hexdigest()
t0 = time.perf_counter()
cracked = next(f"98765{i:05d}" for i in range(100000) if hashlib.sha256(f"98765{i:05d}".encode()).hexdigest() == leaked_hash)
print(f"\nUnsalted SHA-256 of a phone number recovered in {time.perf_counter() - t0:.2f}s (we only guessed the last 5 digits; all 10-digit mobiles fit in ~10^9 hashes - minutes on a GPU). Recovered: {cracked}")

### Your turn 6 - extend the detector
Indian tax IDs (PAN) have the shape **5 letters, 4 digits, 1 letter** (e.g. `ABCDE1234F`). Add a `"pan"` pattern to `PII_PATTERNS` and verify that `detect_pii` finds it in a sentence, without flagging the ordinary word `CLAIMANT`.

In [ ]:
# TODO: PII_PATTERNS["pan"] = r"..."      # HINT: \b[A-Z]{5}\d{4}[A-Z]\b
# test = "Claimant PAN is ABCDE1234F; claim ref CLAIMANT1234."
# print(detect_pii(test))

## 6.4 k-anonymity: can "harmless" columns re-identify someone?
Suppose ACME shares a claims extract with a research partner after removing names. Remaining: age, pincode, gender and the *sensitive* attribute (diagnosis). These **quasi-identifiers** can be combined with public data (voter rolls, social media) to re-identify people -
a famous historical example re-identified a state governor's health record from just ZIP code, birth date and sex.

**k-anonymity**: every combination of quasi-identifiers must be shared by at least *k* records, so an individual hides in a crowd of at least k.
We measure it, generalise (coarser age band, shorter pincode), suppress what remains, and also peek at **l-diversity** (a crowd where everybody has the same diagnosis still leaks it).

In [ ]:
rr = np.random.default_rng(5); n = 300
toy = pd.DataFrame({"age": rr.integers(20, 70, n), "pincode": rr.choice(["400001", "400053", "400072", "560001", "560034", "110001", "110024", "600001"], n),
                    "gender": rr.choice(["F", "M"], n), "diagnosis": rr.choice(["diabetes", "hypertension", "asthma", "fracture", "none"], n, p=[.2, .25, .15, .1, .3])})
QI = ["age", "pincode", "gender"]
def k_anonymity(df, qi): sizes = df.groupby(qi).size(); return int(sizes.min()), int((df.groupby(qi)[qi[0]].transform("size") < 5).sum())
k0, risky0 = k_anonymity(toy, QI); print(f"Raw extract:         k = {k0}  | records in groups smaller than 5: {risky0} of {len(toy)}")

gen = toy.assign(age=(toy.age // 10 * 10).astype(str) + "s", pincode=toy.pincode.str[:3] + "xxx")                 # generalise
k1, risky1 = k_anonymity(gen, QI); print(f"After generalising:  k = {k1}  | records in groups smaller than 5: {risky1}")
sizes = gen.groupby(QI)[QI[0]].transform("size"); released = gen[sizes >= 5]                                      # suppress remaining small groups
k2, _ = k_anonymity(released, QI); print(f"After suppression:   k = {k2}  | released {len(released)} of {len(toy)} records ({len(released) / len(toy):.0%})  <- the utility cost of privacy")
ldiv = released.groupby(QI).diagnosis.nunique(); print(f"l-diversity check: min distinct diagnoses per group = {ldiv.min()} (if this were 1, group membership alone would reveal the diagnosis)")

For stronger guarantees (and for ML specifically) the toolbox grows: **differential privacy** (calibrated noise, formal guarantee), **federated learning** (train without centralising raw data), **synthetic data** (with privacy testing - not automatically safe) and **confidential computing**. Know they exist and what problem each solves; you will rarely hand-implement them.

## 6.5 Access control as code: role -> column policy
Least privilege means people and services get **only what their job needs**. A policy engine decides per role: which columns are visible in clear, masked, pseudonymised or dropped - and which rows (e.g. a regional analyst only sees their region).
Two design principles: **default deny** (a column not mentioned is not visible) and **decide close to the data** (enforced in the warehouse/lakehouse layer, not in each dashboard).

In [ ]:
pii_frame = claims[["claim_id", "customer_name", "phone", "email", "region", "loss_type", "claim_amount", "is_fraud", "notes"]].head(2000).copy()
ROLE_POLICY = {
    "fraud_investigator": {"columns": {"claim_id": "clear", "customer_name": "clear", "phone": "mask_last4", "email": "clear", "region": "clear", "loss_type": "clear", "claim_amount": "clear", "is_fraud": "clear", "notes": "clear"}},
    "data_scientist":     {"columns": {"claim_id": "clear", "customer_name": "drop", "phone": "drop", "email": "pseudonymise", "region": "clear", "loss_type": "clear", "claim_amount": "clear", "is_fraud": "clear", "notes": "redact"}},
    "south_region_analyst": {"columns": {"claim_id": "clear", "region": "clear", "loss_type": "clear", "claim_amount": "clear"}, "row_filter": lambda d: d.region == "South"},
}
def apply_policy(df, role):
    if role not in ROLE_POLICY: raise PermissionError(f"role '{role}' has no policy: default deny")
    pol = ROLE_POLICY[role]; d = df[pol["row_filter"](df)] if "row_filter" in pol else df
    out = pd.DataFrame(index=d.index)
    for col, action in pol["columns"].items():                               # any column NOT listed is dropped (default deny)
        if action == "clear": out[col] = d[col]
        elif action == "mask_last4": out[col] = d[col].str.replace(r"\d(?=\d{4})", "X", regex=True)
        elif action == "pseudonymise": out[col] = d[col].map(lambda v: pseudonymise(v, col))
        elif action == "redact": out[col] = d[col].map(redact)
    return out

for role in ROLE_POLICY:
    v = apply_policy(pii_frame, role); print(f"\n=== {role}: {len(v)} rows x {v.shape[1]} cols ==="); print(v.drop(columns=[c for c in ["notes"] if c in v.columns]).head(3).to_string(index=False))
print("\nA data scientist's notes are redacted:", apply_policy(pii_frame, "data_scientist").notes.iloc[0][:150], "...")
print("LOOK CLOSELY: the phone and email are gone, but the customer's NAME is still in the text. Regexes find things with a fixed shape; names, addresses and free-form identifiers need NER models or an LLM-based PII scanner - and a human sampling the output.")
try: apply_policy(pii_frame, "intern")
except PermissionError as e: print("\nDenied ->", e)

## 6.6 The regulatory landscape (overview, not legal advice)
Governance exists because regulations make *someone accountable* for data. Names differ; the themes recur. Treat this as a map to start the right conversation with your legal and compliance partners - **it is not legal advice, and obligations depend on jurisdiction, sector and use case.**

| Regime (examples) | Typical scope | Themes that touch AI data foundations |
|---|---|---|
| **GDPR** (EU/UK-style) | Personal data of individuals in the EU | Lawful basis; **purpose limitation**; **data minimisation**; storage limitation; rights of access/erasure; DPIAs for high-risk processing; safeguards around solely automated decisions with significant effects; cross-border transfer rules |
| **India DPDP Act 2023** | Digital personal data in India | Consent and notice; purpose limitation; duties of the "data fiduciary"; grievance redressal; breach reporting; additional duties for significant fiduciaries |
| **RBI directions** (Indian banks/NBFCs/payment firms) | Regulated financial entities | Storage of payment-system data within India; IT governance, outsourcing and cloud-risk expectations; cyber-security and audit trails; (insurers have analogous IRDAI information-security guidelines) |
| **HIPAA** (US health) | Protected health information held by covered entities and their business associates | "Minimum necessary" access; Privacy and Security Rules; de-identification standards (Safe Harbor / Expert Determination); audit logging; business-associate agreements with vendors |
| **AI-specific rules** (e.g. EU AI Act) | Risk-tiered AI systems | For high-risk systems: documented **data governance**, representativeness and bias examination, traceability |

**Design consequence for architects:** build once for the strictest regime you operate under - catalogue + classification tags, residency-aware storage, retention automation, deletion workflows that reach derived features, audit logs, and consent/purpose metadata that travels with the data.

---
# Part 7 - Data architecture for AI systems

Time to assemble the parts. Here is the reference architecture most enterprise AI platforms converge on. Read it left to right as the **journey of one fact** - from a customer's action to a model's decision - and top-to-bottom as three *planes* that cut across everything.

**Data flow (left -> right)**
1. **Sources** - core systems (structured), apps (semi-structured), documents/images (unstructured), IoT (streams).
2. **Ingestion** - *batch/CDC* for tables and files; a *durable event log* (Kafka/Kinesis/Event Hubs style) for streams. The log is the replay button.
3. **Lakehouse, medallion layers**
   * **Bronze (raw)** - exact copy, append-only, immutable, schema-on-read. Your insurance policy and audit trail.
   * **Silver (curated)** - cleaned, deduplicated, typed, conformed keys, **quality gates enforced** (Part 4), PII tagged/masked (Part 6).
   * **Gold (consumption)** - business-level marts and **feature tables** with event timestamps for point-in-time correctness (Part 3).
4. **Serving** - *feature store* (offline for training, online low-latency key-value for inference), *warehouse/SQL* for BI, a *vector index* for semantic search and RAG (Lab 3).
5. **Consumers** - training pipelines, real-time scoring APIs, GenAI apps, dashboards.

**Cross-cutting planes**
* **Orchestration & delivery** - schedules, dependencies, retries, backfills, data contracts, infrastructure-as-code, CI/CD for pipelines *and* models.
* **Governance** - catalogue, lineage, classification, access policies, retention, audit (Part 6).
* **Observability** - freshness, volume and schema monitors, quality scorecards, feature drift and train/serve skew, cost and SLO dashboards.

Below we draw it with matplotlib patches - note the small **Q** markers: quality gates sit *between* layers, because bad data is cheapest to stop at the door.

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(14, 8.2)); ax.set_xlim(0, 100); ax.set_ylim(0, 62); ax.axis("off")
def box(x, y, w, h, text, fc="white", ec=INK, tc=INK, fs=8.2, lw=1.4, bold=False):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.25,rounding_size=0.8", fc=fc, ec=ec, lw=lw))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fs, color=tc, fontweight="bold" if bold else "normal", linespacing=1.25)
def arrow(p, q, color=INK, style="-|>", ls="-", lw=1.4, rad=0.0):
    ax.add_patch(FancyArrowPatch(p, q, arrowstyle=style, mutation_scale=11, color=color, lw=lw, ls=ls, connectionstyle=f"arc3,rad={rad}"))
def col_title(x, w, text): ax.text(x + w / 2, 49.2, text, ha="center", fontsize=9.5, fontweight="bold", color=TEAL)

# --- planes (bands) ---
box(1, 53.5, 98, 5.5, "ORCHESTRATION & DELIVERY   |   workflows, retries, backfills   |   data contracts   |   infrastructure-as-code   |   CI/CD for pipelines and models", fc=INK, tc="white", fs=8.5, bold=True)
box(1, 7.0, 98, 4.6, "GOVERNANCE PLANE   |   catalogue   |   lineage   |   PII classification and masking   |   access policies (RBAC/ABAC)   |   retention and deletion   |   audit log", fc=GOLD, ec=INK, tc=INK, fs=8.5, bold=True)
box(1, 1.0, 98, 4.6, "OBSERVABILITY   |   freshness and volume monitors   |   schema drift   |   quality scorecards   |   feature drift and train/serve skew   |   cost and SLO dashboards", fc=CORAL, ec=INK, tc="white", fs=8.5, bold=True)

# --- columns ---
for x, w, t in [(0.5, 13, "SOURCES"), (17, 13, "INGESTION"), (34, 33, "LAKEHOUSE (medallion)"), (72, 13, "SERVING"), (89, 10.5, "CONSUMERS")]: col_title(x, w, t)
src = ["Core systems\n(claims, policy,\npayments)  [structured]", "Mobile / web apps\n(JSON events)\n[semi-structured]", "Documents, photos,\nemail, call audio\n[unstructured]", "Telematics / IoT\nsensors  [streaming]"]
sy = [41, 33.5, 26, 18.5]
for t, y in zip(src, sy): box(0.5, y, 13, 6, t, fc="#E8F3F4", fs=7.3)
box(17, 34, 13, 9, "Batch / CDC\ningestion\n(files, tables,\nobject drops)", fc="#DDEFC0", fs=8)
box(17, 19.5, 13, 9, "Event log / stream\nprocessing\n(windows, late data,\ndedupe)", fc="#DDEFC0", fs=7.8)
for (bx, nm, sub, fc) in [(34, "BRONZE", "raw, immutable,\nreplayable", "#E7C9A9"), (46, "SILVER", "cleaned, conformed,\nquality-gated,\nPII-tagged", "#D9DEE2"), (58, "GOLD", "marts + feature\ntables (as-of)", "#F6DE9B")]:
    box(bx, 24, 9, 15, f"{nm}\n\n{sub}", fc=fc, fs=7.3)
ax.text(50.5, 41.2, "Delta / Iceberg / Parquet on object storage", ha="center", fontsize=7.5, style="italic", color=INK)
serve = [("Feature store\nOFFLINE (training)", 38, 7), ("Feature store\nONLINE (<10 ms)", 29.5, 7), ("Warehouse / SQL\nsemantic layer", 21, 7), ("Vector index\n(search, RAG)  Lab 3", 12.8, 6.6)]
for t, y, h in serve: box(72, y, 13, h, t, fc="#E8F3F4", fs=7.6)
cons = ["Model training\nand evaluation", "Real-time scoring\nAPI (fraud, pricing)", "BI dashboards,\nregulatory reports", "GenAI / RAG\nassistants"]
for t, (_, y, h) in zip(cons, serve): box(89, y + h / 2 - 3, 10.5, 6, t, fc="#E8F3F4", fs=7.3)

# --- flows ---
for y in sy[:3]: arrow((13.7, y + 3), (16.8, 38.5))
arrow((13.7, sy[3] + 3), (16.8, 24)); arrow((13.7, sy[1] + 3), (16.8, 26), color=TEAL, ls="--")
arrow((30.2, 38.5), (33.8, 35)); arrow((30.2, 24), (33.8, 28))
arrow((43.2, 31.5), (45.8, 31.5)); arrow((55.2, 31.5), (57.8, 31.5))
for qx in (44.5, 56.5): ax.add_patch(plt.Circle((qx, 35.3), 0.9, fc=GOLD, ec=INK, lw=1.2, zorder=5)); ax.text(qx, 35.2, "Q", ha="center", va="center", fontsize=7, fontweight="bold", zorder=6)
for _, y, h in serve: arrow((67.2, 31.5), (71.8, y + h / 2), rad=0.0)
for _, y, h in serve: arrow((85.2, y + h / 2), (88.8, y + h / 2))
ax.text(50.5, 46.0, "Q = quality gate: stop or quarantine bad data between layers", ha="center", fontsize=8, color=INK)
ax.set_title("ACME reference architecture for AI data foundations", fontsize=14, fontweight="bold", pad=6)
plt.tight_layout(); fig.savefig(ART / "reference_architecture.png", dpi=130, bbox_inches="tight"); plt.show()

### Why this shape? (the architect's reasoning, in five lines)
* **Raw is sacred (Bronze):** you can always rebuild Silver and Gold when a rule changes or a bug is found - you cannot rebuild what you never kept.
* **Quality gates between layers** turn "the dashboard looks odd" into "the Silver job failed a freshness rule at 02:14" - detection moves from days to minutes.
* **One feature definition, two stores** (offline/online) is how you kill train/serve skew by construction.
* **Governance and observability are planes, not phases:** retrofitting lineage or PII tagging after go-live is 10x harder than tagging at ingestion.
* **Batch and streaming share the storage layer** (lakehouse tables), so Lambda-style duplication is limited to the few features that truly need seconds-level freshness.

## 7.2 Design exercise: a fraud decision in under 200 ms

**Scenario.** ACME's digital-payments arm wants to **approve or block a claim payout / card transaction in under 200 ms (p99)**. Signals needed: velocity ("how many payouts to this account in the last 5 minutes / 24 hours?"), the customer's claim history, and device telemetry.
The model is **retrained nightly**. Investigator outcomes (the labels) arrive **up to 30 days later**. The data includes card numbers and national IDs. The service must be 99.9% available.

Fill in the decision table below - one short phrase per row. A tiny checker looks for the key *concepts* (not exact words) so you get instant feedback. There is more than one valid architecture; justify yours aloud.

In [ ]:
DESIGN_RUBRIC = {   # row -> acceptable concept keywords (any one matches, case-insensitive)
    "event_ingestion":        ["stream", "kafka", "event log", "kinesis", "event hub"],
    "raw_storage":            ["bronze", "object", "lakehouse", "append", "immutable", "lake"],
    "online_features":        ["online", "key-value", "redis", "low-latency", "low latency", "sliding", "streaming aggregat"],
    "training_data":          ["offline", "point-in-time", "as-of", "point in time", "lakehouse", "parquet"],
    "training_cadence":       ["nightly", "batch", "daily", "scheduled"],
    "label_handling":         ["delay", "late", "maturity", "join later", "backfill", "30 day", "lag"],
    "quality_gates":          ["freshness", "schema", "null", "skew", "drift", "gate", "contract", "monitor"],
    "security_and_privacy":   ["token", "mask", "encrypt", "pci", "rbac", "least privilege", "pseudonym", "audit", "vault"],
}
DESIGN_PROMPTS = {
    "event_ingestion":      "How do payment/claim events reach the platform?",
    "raw_storage":          "Where is the immutable raw copy kept?",
    "online_features":      "How are velocity features (5 min / 24 h) computed and served within the latency budget?",
    "training_data":        "Where does the training set come from, and how do you avoid leakage?",
    "training_cadence":     "Training cadence and pattern?",
    "label_handling":       "Labels arrive up to 30 days late - how does the pipeline cope?",
    "quality_gates":        "Which automated checks protect training AND serving?",
    "security_and_privacy": "How are card numbers and IDs protected end to end?",
}
def check_design(design):
    rows = []
    for k, kws in DESIGN_RUBRIC.items():
        txt = str(design.get(k, "")).lower()
        rows.append({"decision": k, "your_answer": design.get(k, "")[:70], "concept_found": "PASS" if any(w in txt for w in kws) else "FAIL"})
    out = pd.DataFrame(rows); print(out.to_string(index=False)); print(f"\nScore: {(out.concept_found == 'PASS').sum()}/{len(out)}"); return out

# TODO: replace each "TODO" with a short phrase (use DESIGN_PROMPTS for the question). HINT: think stream vs batch, online vs offline store, point-in-time joins, delayed labels, data contracts, tokenisation.
my_design = {k: "TODO" for k in DESIGN_RUBRIC}
for k, q in DESIGN_PROMPTS.items(): print(f"{k:<22} {q}")
_ = check_design(my_design)

---
# Part 8 - Wrap-up

## What you did in this lab
* Handled **structured, semi-structured, unstructured and streaming** data, and saw that each is stored and queried differently.
* Built a **batch ETL** (idempotent, with quarantine) and a **streaming window pipeline** (watermarks, late data, dedupe), and quantified the freshness-completeness trade-off.
* Engineered **features** with measurable lift, then *caused and detected* **train/serve skew** and **point-in-time leakage**.
* Wrote **data-quality rules**, injected defects, built a **scorecard**, and measured how label noise and missing data erode a model.
* Measured **annotator agreement**, aggregated labels, and used **active learning** to cut labelling cost.
* Implemented **catalogue linting, PII detection, pseudonymisation, k-anonymity and role-based column policy**.
* Drew and defended a **reference architecture** for AI data foundations.

## Key takeaways
1. **Data shape drives storage and processing** - raw stays raw; curated becomes tables; streams stay on a replayable log.
2. **Buy only the freshness the decision needs.** Real-time is a cost centre; use it surgically.
3. **Idempotency + dedupe** turn at-least-once delivery into "effectively exactly-once".
4. **One feature definition, used offline and online.** Everything else breeds skew.
5. **Be suspicious of great offline numbers** - check for leakage and as-of correctness.
6. **Quality is tested continuously and gates pipelines;** the model will not tell you the data went bad.
7. **Labels are opinions with error bars.** Measure agreement, adjudicate disputes, label the informative items first.
8. **Governance is code:** catalogue, classification, masking, least-privilege access, retention - enforced by the platform, not by memory.
9. **Free text hides PII;** regex finds some, never all.
10. **Governance and observability are planes across the architecture, not phases at the end.**

## The architect's checklist: 10 questions to ask about any AI data foundation
1. **Purpose & value:** Which decision does this data support, and how fresh must it be to change that decision?
2. **Sources & rights:** Where does each dataset originate, who owns it, and do we have a lawful basis/consent for *this* use?
3. **Shape & storage:** Is each dataset stored in the form that suits how it is queried (raw object store, tables, log, vector index)?
4. **Pipelines:** Which paths are batch and which are streaming - and why? Are they idempotent, replayable and backfill-able?
5. **Quality:** What automated checks run, at which layer, and what happens when one fails (who is paged, what is blocked)?
6. **Features:** Is there one definition shared by training and serving? Are features point-in-time correct and versioned?
7. **Labels:** Where does ground truth come from, how noisy is it (kappa?), and how fast does it arrive?
8. **Governance & security:** Is PII classified and minimised? Who can see what, and can we prove it (audit)? How do deletion requests propagate to features and models?
9. **Observability:** Do we monitor freshness, volume, drift and train/serve skew - and the cost per pipeline?
10. **Operability:** Who owns it at 3 a.m.? What are the SLOs, the recovery plan, and the exit path if a vendor changes?

## Homework and stretch tasks
1. **Streaming:** extend `run_stream` with *session windows* (a trip ends after 5 minutes of silence per device) and compare with fixed windows.
2. **Features:** add a *time-decayed* claim count (half-life 30 days) as a registered feature, and check it is point-in-time correct with an as-of test.
3. **Quality:** add a *distribution drift* rule (population stability index or a KS test from `scipy.stats`) comparing this month's `claim_amount` with the previous 3 months; try it on the unit-mismatch partition.
4. **Labeling:** simulate **correlated** annotators (they err on the same hard items) and see how much majority vote degrades versus the independent case.
5. **Governance:** write a *deletion workflow*: given a `customer_id`, list every dataset in the catalogue (via lineage) that must be purged or re-derived.
6. **Architecture:** redraw the diagram for a *batch-only* ACME (no streaming) - what business capabilities do you lose, and what do you save?
7. **Preview of Lab 3:** replace the keyword features with *text embeddings* of the adjuster notes and compare the AUC lift - that is where unstructured data becomes first-class.

In [ ]:
# Housekeeping: close the SQLite connection, remove the temp landing zone, list what we kept.
import shutil
for c_ in (conn, naive):
    try: c_.close()
    except Exception: pass
shutil.rmtree(WORK, ignore_errors=True)
print("temp landing zone removed.  Saved artifacts:")
for p_ in sorted(ART.glob("*.png")): print("  ", p_)
print("\nLab 2 complete. Next: Lab 3 - embeddings, vector search and unstructured data.")